# Analisis Telemetri DNS Skala Besar (11,7 Juta Pesan)
> Pipeline Analisis Operasional & Rekayasa Data Trafik DNS Otoritatif

## Konteks & Sumber Data
- **Dataset**: Log telemetri paket DNS otoritatif (~2,3 GB, 11,7 juta record) selama jendela 30 menit.
- **Tujuan**: Profiling performa protokol, evaluasi kepatuhan RFC (RFC 1035, RFC 2308, RFC 6891), analisis konsentrasi beban (Pareto prefix /48), dekomposisi lonjakan anomali respons (NXDOMAIN), dan analisis truncation EDNS.
- **Public Suffix List**: Pemetaan registrable domain memakai paket `publicsuffix2` (snapshot bawaan paket).


In [ ]:
# ---------------------------------------------------------------
# [Langkah 1] Batasi thread numerik sebelum import numpy
# Tujuan  : membatasi thread OpenMP/MKL menjadi 4
# Alasan  : variabel lingkungan hanya dibaca saat numpy diimpor, jadi diset sebelum import agar efektif
# CATATAN: ini mitigasi risiko OOM, bukan jaminan bebas OOM
# ---------------------------------------------------------------
import gc
import ipaddress
import math
import os
import re
import sys
from collections import Counter

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

import matplotlib.dates as mdates
import matplotlib.pyplot as plt  # dipakai mulai Cell 17 (visualisasi)
import numpy as np
import pandas as pd
import publicsuffix2  # Public Suffix List (snapshot bawaan paket, offline)

try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------------------------------------------------------
# [Langkah 2] Reproduksibilitas dan lokasi dataset
# Tujuan  : menetapkan seed acak global dan path input CSV
# Alasan  : seed yang sama membuat split dan model dapat direproduksi; assert gagal cepat bila input tidak ada
# ---------------------------------------------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = '../data/sample-dns-30min.csv'
assert os.path.exists(DATA_PATH), f"Dataset tidak ditemukan: {DATA_PATH}"

# ---------------------------------------------------------------
# [Langkah 3] Konfigurasi tampilan tabel
# Tujuan  : menampilkan semua kolom dan teks panjang tanpa terpotong
# Alasan  : tabel hasil analisis adalah deliverable, termasuk sel teks panjang pada tabel keputusan
# ---------------------------------------------------------------
# Konfigurasi display output tabel
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)  # teks panjang (tabel keputusan) tidak terpotong
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("=== 0. ENVIRONMENT & ANTI-OOM CONFIGURATION ===")
print(f"Python      : {sys.version.split()[0]}")
print(f"Pandas      : {pd.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Dataset     : {DATA_PATH}")

# ---------------------------------------------------------------
# Fungsi validasi & agregasi (diinline dari modul perbaikan review eksternal;
# notebook self-contained untuk paket Run All, tanpa impor berkas lokal)
# ---------------------------------------------------------------
def canonical_minutes(ts):
    """Bin 1 menit di dalam rentang timestamp; bukan bukti capture tanpa loss."""
    if ts.empty or ts.isna().any():
        raise ValueError('Timestamp kosong/tidak valid.')
    start = ts.min().ceil('min')
    last = ts.max().floor('min') - pd.Timedelta(minutes=1)
    idx = pd.date_range(start, last, freq='min')
    if len(idx) == 0:
        raise ValueError('Tidak ada interval satu menit penuh.')
    return idx

def validate_dns_frame(df):
    """Validasi kolom inti; mutasi hanya ts_iso menjadi UTC.

    Timestamp tanpa zona ditafsirkan UTC sesuai kontrak dataset ini.
    Jika dataset lain memakai waktu lokal, tentukan zonanya sebelum fungsi ini.
    """
    required = {'ts_iso', 'qr', 'rcode', 'proto', 'tc', 'qname',
                'src_ip', 'dst_ip', 'frame_len', 'qtype_name'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Kolom inti hilang: {sorted(missing)}')
    if df.empty or not df.index.is_unique:
        raise ValueError('Data harus berisi baris dengan index unik.')
    ts = pd.to_datetime(df['ts_iso'], errors='coerce', utc=True)
    bad = ts.isna()
    if bad.any():
        raise ValueError(f'{int(bad.sum())} timestamp kosong/tidak valid; '
                         'audit sumber sebelum agregasi.')
    for col, allowed in [('qr', [0, 1]), ('tc', [0, 1]),
                         ('proto', ['udp', 'tcp'])]:
        if not df[col].isin(allowed).all():
            raise ValueError(f'{col} kosong atau di luar {allowed}.')
    resp = df['qr'].eq(1)
    if df.loc[resp, 'rcode'].isna().any():
        raise ValueError('RCODE respons kosong; denominator perlu diaudit.')
    if df.loc[df['qr'].eq(0), 'src_ip'].isna().any():
        raise ValueError('IP sumber query kosong.')
    if df.loc[resp, 'dst_ip'].isna().any():
        raise ValueError('IP tujuan respons kosong.')
    df['ts_iso'] = ts

def temporal_metrics(df):
    """Panggil setelah validate_dns_frame; tidak menyalin seluruh DataFrame."""
    minute = df['ts_iso'].dt.floor('min')
    idx = canonical_minutes(df['ts_iso'])
    q = df['qr'].eq(0)
    r = df['qr'].eq(1)
    udp_r = r & df['proto'].eq('udp')
    masks = {'packets': pd.Series(True, index=df.index), 'queries': q,
             'responses': r, 'nx': r & df['rcode'].eq(3),
             'udp_resp': udp_r, 'udp_tc': udp_r & df['tc'].eq(1),
             'tcp_packets': df['proto'].eq('tcp')}
    out = pd.DataFrame(index=idx)
    for name, mask in masks.items():
        out[name] = minute.loc[mask].value_counts().reindex(idx, fill_value=0)
    out['nx_rate'] = 100 * out['nx'] / out['responses'].replace(0, np.nan)
    out['tc_ratio'] = 100 * out['udp_tc'] / out['udp_resp'].replace(0, np.nan)
    out['tcp_ratio (%)'] = 100 * out['tcp_packets'] / out['packets'].replace(0, np.nan)
    out['throughput_mib'] = df['frame_len'].groupby(minute).sum(min_count=1).reindex(idx) / 2**20
    # Empty bins carry zero volume but undefined rates. Do not fill rates with 0.
    out.loc[out['packets'].eq(0), 'throughput_mib'] = 0
    return out

def priority_sources(q_work, r_view):
    """q_work: prefix, minute. r_view: prefix, rcode. Keduanya window sama.

    Union mencakup prefix yang hanya muncul pada query atau hanya respons.
    """
    for data, cols in [(q_work, ['prefix', 'minute']), (r_view, ['prefix', 'rcode'])]:
        if data[cols].isna().any().any():
            raise ValueError(f'Nilai kosong pada {cols}; jangan diam-diam dibuang.')
    qn = q_work.groupby('prefix', observed=True).size().rename('query')
    active = q_work.groupby('prefix', observed=True)['minute'].nunique().rename('menit_aktif')
    rn = r_view.groupby('prefix', observed=True).size().rename('respons')
    nx = r_view.loc[r_view['rcode'].eq(3)].groupby('prefix', observed=True).size().rename('nxdomain')
    out = pd.concat([qn, active, rn, nx], axis=1).fillna(0).astype('int64')
    out.index.name = 'prefix'
    out['nx_rate_pct'] = 100 * out['nxdomain'] / out['respons'].replace(0, np.nan)
    total_q = out['query'].sum()
    out['share_pct'] = 100 * out['query'] / total_q if total_q else np.nan
    if int(out['query'].sum()) != len(q_work) or int(out['respons'].sum()) != len(r_view):
        raise ValueError('Rekonsiliasi volume sumber gagal.')
    return out

def extend_anon_map(mapping, prefixes):
    """Tambahkan label unik; pertahankan label lama agar tabel tetap cocok."""
    if len(set(mapping.values())) != len(mapping):
        raise ValueError('Pemetaan lama sudah memiliki label duplikat.')
    if any(re.fullmatch(r'S[1-9][0-9]*', v) is None for v in mapping.values()):
        raise ValueError('Label harus berbentuk S1, S2, ...')
    next_n = max([int(v[1:]) for v in mapping.values()], default=0) + 1
    for prefix in sorted(set(prefixes) - set(mapping)):
        mapping[prefix] = f'S{next_n}'
        next_n += 1
    return mapping

def nx_delta(nx_rows, minute_index, peak, group='dst_ip'):
    """nx_rows hanya respons RCODE=3, memiliki minute dan kolom group.

    Pembanding memasukkan menit nol NX; delta negatif dipertahankan.
    """
    idx = pd.DatetimeIndex(minute_index)
    if idx.has_duplicates or peak not in idx or len(idx) < 2:
        raise ValueError('Window harus unik, memuat puncak, dan minimal 2 menit.')
    work = nx_rows.loc[nx_rows['minute'].isin(idx)]
    if work[group].isna().any():
        raise ValueError('Kelompok NX kosong; audit sebelum dekomposisi.')
    at_peak = work['minute'].eq(peak)
    n_other = len(idx) - 1
    a = work.loc[at_peak].groupby(group, observed=True).size()
    b = work.loc[~at_peak].groupby(group, observed=True).size() / n_other
    union = a.index.union(b.index)
    delta = (a.reindex(union, fill_value=0) - b.reindex(union, fill_value=0)).sort_values(ascending=False)
    peak_total = int(at_peak.sum())
    baseline = int((~at_peak).sum()) / n_other
    net = peak_total - baseline
    if not np.isclose(delta.sum(), net, rtol=1e-10, atol=1e-8):
        raise ValueError('Rekonsiliasi delta gagal.')
    share2 = 100 * delta.head(2).sum() / net if net > 0 else np.nan
    return delta, {'peak_nx': peak_total, 'baseline_per_minute': baseline,
                   'net_delta': net, 'top2_share_net_pct': share2,
                   'n_other_minutes': n_other}



## Cell 2: Cell 2: Schema Definition & In-Memory Fast Load (C-Engine)
> Membaca seluruh 11,7 juta baris menggunakan Pandas Nullable Integers (UInt16, Int8, dsb.) untuk mencegah kegagalan baca akibat nilai kosong (NaN) tanpa membebani RAM.

In [ ]:
from IPython.display import display
print("Memuat dataset ke RAM...")

# ---------------------------------------------------------------
# [Langkah 1] Definisi skema dtype
# Tujuan  : memetakan tiap kolom ke tipe nullable atau kategori yang tepat
# Alasan  : nullable int menampung NA tanpa turun ke float, kategori menghemat RAM untuk kolom berulang
# ---------------------------------------------------------------
dtypes = {
    'ip_ver': 'Int8',
    'proto': 'category',
    'src_port': 'UInt16',
    'dst_port': 'UInt16',
    'frame_len': 'UInt32',
    'dns_len': 'UInt32',
    'dns_id': 'UInt16',
    'qr': 'Int8',          # 0: Query, 1: Response
    'opcode': 'Int8',
    'aa': 'Int8',
    'tc': 'Int8',
    'rd': 'Int8',
    'ra': 'Int8',
    'ad': 'Int8',
    'cd': 'Int8',
    'rcode': 'Int16',
    'qdcount': 'UInt16',
    'ancount': 'UInt16',
    'nscount': 'UInt16',
    'arcount': 'UInt16',
    'qtype': 'UInt16',
    'qtype_name': 'category',
    'qclass': 'UInt16',
    'edns': 'Int8',            # 0/1 (terverifikasi tanpa NA pada seluruh file)
    'edns_udpsize': 'UInt16',  # ukuran buffer integer; NA utk paket non-EDNS
    'do': 'Int8'               # 0/1 (terverifikasi tanpa NA pada seluruh file)
}

# ---------------------------------------------------------------
# [Langkah 2] Muat CSV dengan C engine
# Tujuan  : memuat seluruh rekaman ke RAM dengan tipe terkendali
# Alasan  : keep_default_na=False mencegah qname literal seperti nan atau NULL berubah menjadi NA; field kosong tetap dikenali lewat na_values
# ---------------------------------------------------------------
df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=['ts_iso'],
    keep_default_na=False,   # cegah qname literal ('nan', 'NULL', dst.) menjadi NaN
    na_values=[''],          # field kosong tetap dikenali sebagai NaN
    engine='c'
)

# ---------------------------------------------------------------
# [Langkah 3] Validasi skema fail-fast
# Tujuan  : memastikan struktur input sesuai sebelum analisis
# Alasan  : struktur yang salah lebih baik gagal di awal daripada menghasilkan angka menyesatkan di tengah
# ---------------------------------------------------------------
# Validasi skema eksplisit: gagal cepat bila struktur input berbeda
required_cols = {'ts', 'ts_iso', 'qr', 'rcode', 'qname', 'src_ip', 'dst_ip', 'proto', 'qtype_name'}
missing_cols = required_cols - set(df.columns)
assert not missing_cols, f"Kolom wajib hilang: {sorted(missing_cols)}"
assert df['qr'].isin([0, 1]).all(), "qr mengandung nilai di luar {0, 1}"
validate_dns_frame(df)  # tolak timestamp rusak, flag di luar kontrak, rcode/IP kosong; ts_iso dipastikan UTC

print("Status: Berhasil dimuat.")
print(f"Dimensi Data: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print("Validasi skema+kontrak: kolom inti; qr/tc/proto valid; ts_iso UTC seluruh baris; rcode & IP relevan terisi.")
print("Satuan data: satu baris = satu pesan DNS (query ATAU respons), bukan satu kunjungan/pengguna.")
display(df.head(2))

## Cell 3: 1. Framing Metodologis & Identifikasi Vantage Point
> Mengevaluasi bukti teknis protokol DNS secara inferensial, mengoreksi semantik flag cd=1, ad=0, dan do=1, serta memisahkan kandidat referral berdasarkan proksi (aa=0; authority section belum diperiksa) dari respon negatif NODATA (aa=1) dengan perhitungan dinamis.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 1. FRAMING METODOLOGIS & IDENTIFIKASI VANTAGE POINT ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Mask query dan respons beserta totalnya
# Tujuan  : menjadi dasar seluruh denominator per arah
# Alasan  : query dan respons dipisah agar proporsi tidak tercampur
# ---------------------------------------------------------------
# Boolean mask hemat RAM (tanpa duplikasi DataFrame)
q_mask = df['qr'] == 0
r_mask = df['qr'] == 1
total_q = q_mask.sum()
total_r = r_mask.sum()

# ---------------------------------------------------------------
# [Langkah 2] Audit flag header
# Tujuan  : membedakan perilaku resolver klien dari sifat authoritative server
# Alasan  : rd, cd, do terbaca dari sisi query; ra, ad dari sisi respons
# ---------------------------------------------------------------
# Audit Flag DNSSEC & Rekursi
rd_pct = (df.loc[q_mask, 'rd'] == 1).mean() * 100
cd_pct = (df.loc[q_mask, 'cd'] == 1).mean() * 100
ra_count = (df.loc[r_mask, 'ra'] == 1).sum()
ad_count = (df.loc[r_mask, 'ad'] == 1).sum()

# ---------------------------------------------------------------
# [Langkah 3] DO=1 pada query ber-EDNS
# Tujuan  : menghitung porsi DNSSEC-aware secara benar
# Alasan  : do bermakna hanya bila EDNS ada, jadi denominatornya query ber-EDNS
# ---------------------------------------------------------------
# Evaluasi do=1 terhadap query ber-EDNS saja
edns_q_mask = q_mask & (df['edns'] == 1)
do_on_edns_pct = (df.loc[edns_q_mask, 'do'] == 1).mean() * 100

# ---------------------------------------------------------------
# [Langkah 4] Disambiguasi referral vs NODATA
# Tujuan  : memisahkan dua jenis respons kosong yang berbeda makna
# Alasan  : keduanya ancount=0 tetapi aa membedakan delegasi vs NODATA; rasio disajikan dua basis, seluruh respons dan dalam grup
# ---------------------------------------------------------------
# Disambiguasi: Referral (aa=0) vs NODATA (aa=1) pada rcode=0, ancount=0, nscount>0
empty_answers_mask = (df['rcode'] == 0) & (df['ancount'] == 0) & (df['nscount'] > 0) & r_mask
pure_referrals = (empty_answers_mask & (df['aa'] == 0)).sum()
nodata_answers = (empty_answers_mask & (df['aa'] == 1)).sum()

# Rasio aman terhadap pembagian nol (bila tidak ada respons)
referral_share = (pure_referrals / total_r) if total_r else 0.0
nodata_share = (nodata_answers / total_r) if total_r else 0.0
n_empty_answers = int(empty_answers_mask.sum())
referral_share_in_group = (pure_referrals / n_empty_answers) if n_empty_answers else 0.0
nodata_share_in_group = (nodata_answers / n_empty_answers) if n_empty_answers else 0.0

# ---------------------------------------------------------------
# [Langkah 5] Tabel bukti inferensial
# Tujuan  : menyandingkan angka empiris dengan interpretasi protokoler
# Alasan  : setiap klaim memakai kata konsisten, bukan terbukti, sesuai batas inferensi flag
# ---------------------------------------------------------------
vantage_evidence = pd.DataFrame({
    'Indikator Protokol': [
        'Query dengan rd=1 (Recursion Desired)',
        'Query dengan cd=1 (Checking Disabled)',
        'Query ber-EDNS dengan do=1 (DNSSEC OK)',
        'Respons dengan ra=1 (Recursion Available)',
        'Respons dengan ad=1 (Authenticated Data)',
        'Respons konsisten dengan referral (ancount=0, nscount>0, aa=0)',
        'Respons konsisten dengan NODATA (ancount=0, nscount>0, aa=1)'
    ],
    'Nilai Empiris': [
        f"{rd_pct:.2f}% ({df.loc[q_mask, 'rd'].sum():,} paket)",
        f"{cd_pct:.2f}% ({df.loc[q_mask, 'cd'].sum():,} paket)",
        f"{do_on_edns_pct:.2f}% (dari query ber-EDNS)",
        f"{ra_count:,} paket",
        f"{ad_count:,} paket",
        f"{pure_referrals:,} paket ({referral_share:.2%})",
        f"{nodata_answers:,} paket ({nodata_share:.2%})"
    ],
    'Interpretasi Protokoler': [
        'Dominasi rd=0: Klien mengirim query iteratif (khas resolver rekursif atau QNAME minimisation)',
        'Konsisten dengan resolver hilir yang melakukan validasi DNSSEC sendiri (CD=1 pada query)',
        'Klien resolver hilir secara aktif mendukung penerimaan payload DNSSEC',
        f'Hampir tidak menawarkan rekursi publik ({ra_count:,} respons ra=1 sebagai pengecualian yang layak diaudit)',
        'Wajar untuk server authoritative (bit AD dikelola oleh validating recursive resolver)',
        'Delegasi otoritas ke Name Server anak berdasarkan bit aa (bukan isi authority section)',
        'Konsisten NODATA (nama ada, record diminta nihil); verifikasi penuh butuh isi authority section (RFC 2308)'
    ]
})

display(vantage_evidence)

print("\nKESIMPULAN VANTAGE POINT (BERBASIS BUKTI INFERENSIAL):")
print("1. Profil lalu lintas konsisten dengan SISI AUTHORITATIVE NAME SERVER untuk zona .id.")
print(f"2. Dari {n_empty_answers:,} respons NOERROR dengan ancount=0 (porsi thd seluruh respons: {referral_share:.2%} referral, {nodata_share:.2%} NODATA),")
print(f"   {referral_share_in_group:.2%} konsisten dengan delegasi referral (aa=0) dan {nodata_share_in_group:.2%} konsisten dengan NODATA (aa=1), berdasarkan flag dan counter; konfirmasi penuh butuh isi authority section (RFC 2308).")

## Cell 4: 2. Data Overview, Kardinalitas & Pola DNS 0x20 Encoding
> Mengaudit integritas pasangan query-respons, menyelidiki penyebab 4.606 nilai NaN pada field qname, serta menjelaskan reduksi kardinalitas QNAME sebagai pola yang konsisten dengan randomisasi kapitalisasi; implementasi mitigasi DNS 0x20 belum terverifikasi.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 2. DATA OVERVIEW, INTEGRITAS & DISCOVERY DNS 0x20 ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Ukuran fisik dan rentang waktu
# Tujuan  : mendokumentasikan beban penyimpanan dan cakupan observasi
# Alasan  : satuan GiB basis biner supaya konsisten dengan cara pandas menghitung memori
# ---------------------------------------------------------------
disk_gb = os.path.getsize(DATA_PATH) / (1024 ** 3)
ram_gb = df.memory_usage(deep=True).sum() / (1024 ** 3)
t_min, t_max = df['ts_iso'].min(), df['ts_iso'].max()

# ---------------------------------------------------------------
# [Langkah 2] Investigasi QNAME kosong
# Tujuan  : mengurai arah dan konteks baris tanpa qname
# Alasan  : NA bisa berasal dari field kosong exporter; cross-check FORMERR menguji konsistensi kolom
# ---------------------------------------------------------------
# 1. Investigasi Baris QNAME Kosong (NaN)
null_qname_rows = df[df['qname'].isnull()]
null_qname_rcodes = null_qname_rows['rcode'].value_counts()

# ---------------------------------------------------------------
# [Langkah 3] Audit pasangan query-respons
# Tujuan  : mengukur query yang tidak mendapat respons
# Alasan  : selisih total query dan total respons hanya indikasi loss atau timeout, bukan bukti
# ---------------------------------------------------------------
# 2. Selisih jumlah query dan respons (bukan pencocokan transaksi;
#    pencocokan butuh dns_id + qname + src/dst dan tetap tak menjamin pasangan)
unanswered_queries = total_q - total_r
unanswered_pct = (unanswered_queries / total_q) * 100 if total_q else 0.0

# ---------------------------------------------------------------
# [Langkah 4] Normalisasi kardinalitas 0x20
# Tujuan  : membuktikan efek case-randomization pada hitungan domain
# Alasan  : kardinalitas mentah menggandakan hitungan sekitar 2x sehingga wajib dinormalisasi
# ---------------------------------------------------------------
# 3. Analisis Normalisasi Kardinalitas & Pola DNS 0x20
raw_unique_q = df['qname'].nunique()
clean_domain_series = df['qname'].dropna().str.rstrip('.').str.lower()
clean_unique_q = clean_domain_series.nunique()
reduction_diff = raw_unique_q - clean_unique_q
sample_variations = df[df['qname'].str.lower().str.rstrip('.') == 'smartconnect.id']['qname'].dropna().unique()

# ---------------------------------------------------------------
# [Langkah 5] Tabel overview
# Tujuan  : merangkum metrik kelayakan data dalam satu tabel
# Alasan  : angka ditarik dari variabel, bukan diketik manual
# ---------------------------------------------------------------
overview_metrics = pd.DataFrame({
    'Parameter': [
        'Ukuran File CSV di Disk',
        'Ukuran Objek di RAM',
        'Total Record (Baris)',
        'Total Fitur (Kolom)',
        'Rentang Waktu Log (ts_iso UTC)',
        'Total Durasi Pengamatan',
        'Kardinalitas QNAME Mentah (df[qname].nunique)',
        'Kardinalitas QNAME Ternormalisasi (lowercase & trim dot)',
        'Selisih Variasi Huruf Tereduksi',
        'Baris dengan QNAME Kosong (NaN)',
        'Selisih Jumlah Query dan Respons'
    ],
    'Nilai': [
        f"{disk_gb:.2f} GiB",
        f"{ram_gb:.2f} GiB",
        f"{len(df):,}",
        f"{df.shape[1]}",
        f"{t_min} s/d {t_max} (UTC)",
        str(t_max - t_min),
        f"{raw_unique_q:,}",
        f"{clean_unique_q:,}",
        f"{reduction_diff:,} entri ({(reduction_diff / raw_unique_q):.2%})",
        f"{len(null_qname_rows):,} baris",
        f"{unanswered_queries:,} ({unanswered_pct:.3f}%)"
    ]
})

display(overview_metrics)

print("\nDistribusi RCODE pada Baris dengan QNAME Kosong (NaN):")
display(null_qname_rcodes.to_frame(name='Jumlah Baris'))
n_null_query = int((null_qname_rows['qr'] == 0).sum())
n_null_resp = int((null_qname_rows['qr'] == 1).sum())
formerr_total = int((df['rcode'] == 1).sum())
print(f"Cross-check arah: qr=0 (query) {n_null_query:,} baris | qr=1 (respons) {n_null_resp:,} baris | Total FORMERR (rcode=1) seluruh data: {formerr_total:,}")
if int((null_qname_rows['rcode'] == 1).sum()) > formerr_total:
    print("PERHATIAN: qname kosong ber-rcode=1 melebihi total FORMERR -> verifikasi definisi kolom ke penyedia data.")

print("\nPola DNS 0x20 Bit-Randomization (Varian Kapitalisasi Acak Domain Tunggal) - konsisten dengan randomisasi kapitalisasi; implementasinya belum terverifikasi pada infrastruktur:")
print(f"  {list(sample_variations[:6])}")
print("  Catatan Metodologis: Seluruh selisih kardinalitas murni berasal dari normalisasi huruf besar-kecil")
print("  dan titik akhir. Pola acak ini konsisten dengan teknik anti-spoofing 0x20 (Dagon et al., 2008),")
print("  yang memperkuat RFC 5452. Jangan gunakan kardinalitas mentah karena menggandakan hitungan riil ~2x.")

## Cell 5: 3. Analisis Temporal Steady-State, Trafik & Korelasi Truncation dengan EDNS 1232
> Mengeliminasi menit parsial batas capture, mengevaluasi strict denominator respons UDP untuk tc=1, serta menghubungkan truncation dengan ukuran record DNSSEC dan rekomendasi buffer EDNS 1232.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 3. ANALISIS OPERASIONAL TRAFIK & KORELASI TRUNCATION ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Resample satu menit
# Tujuan  : mengagregasi paket, tcp, tc, dan byte per menit
# Alasan  : Grouper berbasis kolom menghindari salinan DataFrame penuh; tcp dan tc dihitung per bin
# ---------------------------------------------------------------
# Agregasi temporal via modul perbaikan: bin menit penuh kanonik;
# tc_ratio kini = tc=1 pada respons UDP / seluruh respons UDP (bukan per pesan);
# bin tanpa pesan memakai volume 0 dengan rasio NaN (tidak diisi 0%)
temporal = temporal_metrics(df)
n_min_obs = len(temporal.index)  # jumlah bin menit penuh, bukan durasi dibulatkan

# ---------------------------------------------------------------
# [Langkah 2] Eliminasi menit parsial boundary
# Tujuan  : menyisakan interval satu menit yang tercakup penuh data
# Alasan  : menit parsial membuat rasio per menit bias; deret hasil menjadi acuan 29 menit kanonik bagi sel sintesis
# ---------------------------------------------------------------
steady_state_temporal = temporal  # temporal_metrics sudah memakai bin menit penuh kanonik

print(f"[A] STATISTIK OPERASIONAL STEADY-STATE ({len(steady_state_temporal)} Menit Penuh, Menit Parsial Dieliminasi):")
display(steady_state_temporal[['packets', 'tcp_ratio (%)', 'nx_rate', 'tc_ratio', 'throughput_mib']].describe().T)
print("Catatan: 'steady-state' = menit parsial boundary dieliminasi, bukan uji stabilitas operasional statistik.")

# ---------------------------------------------------------------
# [Langkah 3] Denominator truncation
# Tujuan  : memisahkan respons UDP dan TCP
# Alasan  : tc=1 bermakna pada respons; UDP dibedakan dari TCP karena TCP tidak mengenal truncation
# ---------------------------------------------------------------
# Denominator yang Benar untuk Truncation (tc=1)
udp_r_mask = r_mask & (df['proto'] == 'udp')
tcp_r_mask = r_mask & (df['proto'] == 'tcp')
total_udp_r = udp_r_mask.sum()

tc_on_udp = (df.loc[udp_r_mask, 'tc'] == 1).sum()
tc_on_tcp = (df.loc[tcp_r_mask, 'tc'] == 1).sum()
tcp_traffic_total = (df['proto'] == 'tcp').sum()

print("\n[B] EVALUASI STRICT DENOMINATOR TRUNCATION (tc=1):")
print(f"  - Total Respons UDP             : {total_udp_r:,}")
print(f"  - Respons UDP dengan tc=1        : {tc_on_udp:,} ({(tc_on_udp / total_udp_r):.2%})")
print(f"  - Respons TCP dengan tc=1        : {tc_on_tcp:,} (Verifikasi: flag TC bernilai 0 pada segmen TCP)")
print(f"  - Total Trafik TCP (Query+Resp)  : {tcp_traffic_total:,} ({(tcp_traffic_total / len(df)):.2%})")

# ---------------------------------------------------------------
# [Langkah 4] Distribusi EDNS per arah
# Tujuan  : memisahkan buffer query dan respons
# Alasan  : konteks requestor dan responder berbeda sehingga tidak boleh dicampur
# ---------------------------------------------------------------
# Distribusi Buffer EDNS per arah (requestor vs responder terpisah)
print("\n[C] TOP EDNS BUFFER SIZE (edns_udpsize) PER ARAH (konteks requestor vs responder terpisah):")
edns_rows = []
for arah, msk, denom in (('Query (qr=0)', q_mask, total_q), ('Respons (qr=1)', r_mask, total_r)):
    dist = df.loc[msk, 'edns_udpsize'].value_counts(dropna=False).head(3)
    for size, cnt in dist.items():
        edns_rows.append({'Arah': arah, 'Buffer Size (Bytes)': str(size), 'Jumlah Paket': int(cnt),
                          'Proporsi arah (%)': (cnt / denom) * 100 if denom else 0.0})
display(pd.DataFrame(edns_rows))
edns_all_top = df['edns_udpsize'].value_counts().head(1)

print("\n[D] KORELASI TRUNCATION DENGAN RECORD TYPE & UKURAN PAYLOAD:")
tc_qtypes = df.loc[udp_r_mask & (df['tc'] == 1), 'qtype_name'].value_counts().head(5)
tc_qtypes_df = pd.DataFrame({
    'Record Type': tc_qtypes.index.astype(str),
    'Total Truncated': tc_qtypes.values,
    'Pangsa dalam Terpotong (%)': (tc_qtypes.values / tc_on_udp) * 100
})
display(tc_qtypes_df)

# ---------------------------------------------------------------
# [Langkah 5] Insight truncation dinamis
# Tujuan  : menghitung dua qtype terpotong teratas dari tabel
# Alasan  : angka dihitung dari data agar tetap valid pada dataset lain
# ---------------------------------------------------------------
top2_tc = tc_qtypes_df.head(2)
top2_desc = ", ".join(f"{row['Record Type']} ({row['Pangsa dalam Terpotong (%)']:.2f}%)" for _, row in top2_tc.iterrows())
print(f"Insight Operasional: Buffer dominan {int(edns_all_top.index[0])} byte ({(edns_all_top.iloc[0] / len(df)) * 100:.2f}% dari seluruh paket) adalah rekomendasi resmi DNS Flag Day 2020.")
print(f"Truncation UDP: {top2_desc} adalah PANGSA di antara respons UDP yang terpotong, bukan rasio terpotong per tipe (rasio per tipe tersedia pada bagian 14).")
print("Rekomendasi: Jangan menaikkan buffer UDP secara sembarangan (risiko fragmentasi IP), melainkan pastikan")
print("infrastruktur TCP selalu siap melayani koneksi fallback tanpa degradasi performa.")

## Cell 6: 4. Analisis DNS Query (qr = 0)
> Mengevaluasi komponen query murni: distribusi Record Type (qtype_name), Opcodes, dan operasional flag rekursi klien.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 4. ANALISIS DNS QUERY (qr = 0) ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Distribusi qtype query
# Tujuan  : menunjukkan tipe record yang paling diminta
# Alasan  : denominator = seluruh query qr=0
# ---------------------------------------------------------------
queries_view = df.loc[q_mask]

# Top Query Types
qtype_counts = queries_view['qtype_name'].value_counts().head(10)
qtype_df = pd.DataFrame({
    'Record Type': qtype_counts.index.astype(str),
    'Jumlah Query': qtype_counts.values,
    'Proporsi (%)': (qtype_counts.values / total_q) * 100
})
print("[A] TOP 10 DNS RECORD TYPE YANG DIMINTA:")
display(qtype_df)

# ---------------------------------------------------------------
# [Langkah 2] Flag operasional query
# Tujuan  : memotret perilaku klien
# Alasan  : rd, cd, dan qdcount>1 menandai iterasi, validasi mandiri, dan anomali protokol
# ---------------------------------------------------------------
# Operasional Flags pada Query
q_flags = pd.DataFrame({
    'Flag Operasional': [
        'Recursion Desired (rd=1)',
        'Checking Disabled (cd=1)',
        'Multiple Questions (qdcount > 1)'
    ],
    'Jumlah Paket': [
        (queries_view['rd'] == 1).sum(),
        (queries_view['cd'] == 1).sum(),
        (queries_view['qdcount'] > 1).sum()
    ],
    'Proporsi (%)': [
        ((queries_view['rd'] == 1).sum() / total_q) * 100,
        ((queries_view['cd'] == 1).sum() / total_q) * 100,
        ((queries_view['qdcount'] > 1).sum() / total_q) * 100
    ]
})
print("\n[B] STATUS FLAG OPERASIONAL QUERY:")
display(q_flags)

## Cell 7: 5. Analisis DNS Response (qr = 1)
> Mengevaluasi distribusi kode balasan server (RCODE), status section counter (ancount, nscount, arcount), serta flag response otoritatif (aa).

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 5. ANALISIS DNS RESPONSE (qr = 1) ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Distribusi RCODE
# Tujuan  : menunjukkan status balasan yang paling sering terjadi
# Alasan  : denominator = seluruh respons; mapping kode dari RFC 1035
# ---------------------------------------------------------------
responses_view = df.loc[r_mask]

# RCODE Mapping
rcode_map = {0: 'NOERROR', 1: 'FORMERR', 2: 'SERVFAIL', 3: 'NXDOMAIN', 4: 'NOTIMP', 5: 'REFUSED', 9: 'NOTAUTH'}
rcode_counts = responses_view['rcode'].value_counts().head(8)
rcode_df = pd.DataFrame({
    'RCODE': rcode_counts.index,
    'Status Respon': [rcode_map.get(c, f"CODE_{c}") for c in rcode_counts.index],
    'Jumlah Paket': rcode_counts.values,
    'Proporsi (%)': (rcode_counts.values / total_r) * 100
})
print("[A] DISTRIBUSI DNS RESPONSE CODES (RCODE):")
display(rcode_df)

# ---------------------------------------------------------------
# [Langkah 2] Statistik section counter
# Tujuan  : membedakan jawaban isi, referral, dan kosong
# Alasan  : ancount>0 berarti ada jawaban; nscount>0 dengan ancount=0 menandai referral
# ---------------------------------------------------------------
# Section Counter Statistik
counter_stats = responses_view[['ancount', 'nscount', 'arcount']].describe().T
counter_stats['Non-Zero (Total)'] = [
    (responses_view['ancount'] > 0).sum(),
    (responses_view['nscount'] > 0).sum(),
    (responses_view['arcount'] > 0).sum()
]
counter_stats['Non-Zero (%)'] = (counter_stats['Non-Zero (Total)'] / total_r) * 100

print("\n[B] STATISTIK SECTION JAWABAN (ancount, nscount, arcount):")
display(counter_stats[['count', 'mean', 'std', 'max', 'Non-Zero (Total)', 'Non-Zero (%)']])

# ---------------------------------------------------------------
# [Langkah 3] Flag respons
# Tujuan  : memotret sifat server dari sisi respons
# Alasan  : tc, aa, ra, ad masing-masing menandai truncation, otoritatif, rekursi, validasi
# ---------------------------------------------------------------
# Flags pada Respons
r_flags = pd.DataFrame({
    'Flag Response': [
        'Truncated Packet (tc=1)',
        'Authoritative Answer (aa=1)',
        'Recursion Available (ra=1)',
        'Authenticated Data (ad=1)'
    ],
    'Jumlah Paket': [
        (responses_view['tc'] == 1).sum(),
        (responses_view['aa'] == 1).sum(),
        (responses_view['ra'] == 1).sum(),
        (responses_view['ad'] == 1).sum()
    ],
    'Proporsi (%)': [
        ((responses_view['tc'] == 1).sum() / total_r) * 100,
        ((responses_view['aa'] == 1).sum() / total_r) * 100,
        ((responses_view['ra'] == 1).sum() / total_r) * 100,
        ((responses_view['ad'] == 1).sum() / total_r) * 100
    ]
})
print("\n[C] KONDISI FLAG RESPONSE:")
display(r_flags)

## Cell 8: 6. Analisis Sumber Subnet Prefix /48 & Bedah Irisan Pareto
> Membedah irisan antara subnet pengirim terbesar (2d68::/48) dengan domain dominan (smartconnect.id), membuktikan bahwa 99,5% query domain tersebut berasal dari satu subnet, serta menyusun rekomendasi mitigasi bertingkat.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 6. ANALISIS SUMBER QUERY (SUBNET PREFIX & PARETO) ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Peta prefix per IP unik
# Tujuan  : mengelompokkan IP sumber ke prefix /48 atau /24
# Alasan  : modul ipaddress aman untuk IPv6 bentuk ringkas; dihitung per IP unik agar murah
# ---------------------------------------------------------------
# 1. Pemetaan Prefix /48 (ipaddress: aman untuk IPv6 bentuk ringkas)
unique_q_ips = df.loc[q_mask, 'src_ip'].dropna().unique()
def fast_prefix(ip):
    try:
        addr = ipaddress.ip_address(ip)
    except ValueError:
        return ip
    prefix_len = 48 if addr.version == 6 else 24
    return str(ipaddress.ip_network(f'{addr}/{prefix_len}', strict=False))

prefix_map = {ip: fast_prefix(ip) for ip in unique_q_ips}

# ---------------------------------------------------------------
# [Langkah 2] Pareto subnet sumber
# Tujuan  : mengurutkan kontributor query terbesar
# Alasan  : agregasi per /48 karena itu unit alokasi yang tipikal
# ---------------------------------------------------------------
# Agregasi Subnet
ip_counts = df.loc[q_mask, 'src_ip'].value_counts()
prefix_counts = Counter()
for ip, count in ip_counts.items():
    prefix_counts[prefix_map[ip]] += count

prefix_df = pd.DataFrame(prefix_counts.most_common(10), columns=['Subnet Prefix', 'Total Query'])
prefix_df['Share (%)'] = (prefix_df['Total Query'] / total_q) * 100
prefix_df['Kumulatif (%)'] = prefix_df['Share (%)'].cumsum()

print(f"[A] KONSENTRASI SUMBER QUERY DARI {len(prefix_map):,} IP UNIK ({len(prefix_counts):,} SUBNET PREFIX):")
display(prefix_df)

print("Catatan: satu prefix /48 tidak otomatis satu organisasi/resolver; verifikasi apakah IP dianonimkan")
print("tanpa mengubah struktur prefix sebelum interpretasi organisasional.")

top_subnet_name = prefix_df.loc[0, 'Subnet Prefix']
top_subnet_share = prefix_df.loc[0, 'Share (%)']

# ---------------------------------------------------------------
# [Langkah 3] Bedah subnet teratas
# Tujuan  : mengurai komposisi trafik kelompok dominan
# Alasan  : domain dan qtype di dalam subnet menjelaskan karakter pemakaiannya
# ---------------------------------------------------------------
# 2. Bedah Irisan Subnet Teratas
queries_df_view = df.loc[q_mask, ['src_ip', 'qname', 'qtype_name', 'cd']].copy()
queries_df_view['prefix'] = queries_df_view['src_ip'].map(prefix_map)

subnet_queries = queries_df_view[queries_df_view['prefix'] == top_subnet_name].copy()
subnet_queries['clean_qname'] = subnet_queries['qname'].dropna().str.rstrip('.').str.lower()

top_domains_in_subnet = subnet_queries['clean_qname'].value_counts().head(5)
top_qtypes_in_subnet = subnet_queries['qtype_name'].value_counts().head(5)
cd_in_subnet = (subnet_queries['cd'] == 1).mean() * 100

print(f"\n[B] KOMPOSISI TRAFIK SUBNET SUMBER TERBESAR ({top_subnet_name}):")
print(f"  - Total Query dari Subnet ini  : {len(subnet_queries):,} ({len(subnet_queries)/total_q:.2%} dari seluruh query)")
print(f"  - Rasio Penggunaan CD=1        : {cd_in_subnet:.2f}%")

print("\nTop 5 Domain yang Diminta oleh Subnet Ini:")
display(pd.DataFrame({
    'Domain': top_domains_in_subnet.index,
    'Total Query': top_domains_in_subnet.values,
    'Proporsi dalam Subnet (%)': (top_domains_in_subnet.values / len(subnet_queries)) * 100
}))

print("\nTop 5 Query Type dalam Subnet Ini:")
display(pd.DataFrame({
    'Record Type': top_qtypes_in_subnet.index.astype(str),
    'Total Query': top_qtypes_in_subnet.values,
    'Proporsi dalam Subnet (%)': (top_qtypes_in_subnet.values / len(subnet_queries)) * 100
}))

# ---------------------------------------------------------------
# [Langkah 4] Pangsa domain dominan
# Tujuan  : menghitung ketergantungan subnet pada satu domain
# Alasan  : domain dihitung dinamis agar tetap valid pada dataset lain
# ---------------------------------------------------------------
# 3. Analisis Pangsa Domain Dominan (dinamis, bukan hardcode)
target_domain = clean_domain_series[df['qr'] == 0].value_counts().index[0]
global_target_queries = (clean_domain_series[df['qr'] == 0] == target_domain).sum()
subnet_target_queries = (subnet_queries['clean_qname'] == target_domain).sum()
target_subnet_share = (subnet_target_queries / global_target_queries) * 100 if global_target_queries else 0.0

print(f"\nPangsa Subnet terhadap Domain Dominan ('{target_domain}'):")
print(f"  - Query '{target_domain}' dari Subnet ini : {subnet_target_queries:,} dari {global_target_queries:,} total ({target_subnet_share:.2f}%)")

print("\nINSIGHT OPERASIONAL & ARSITEKTUR:")
print(f"Subnet {top_subnet_name} konsisten dengan infrastruktur resolver rekursif tersentralisasi.")
top3_share_in_subnet = (top_domains_in_subnet.iloc[:3].sum() / len(subnet_queries)) * 100 if len(subnet_queries) else 0.0
print(f"Tiga domain teratas menyerap {top3_share_in_subnet:.2f}% kueri subnet, di antaranya domain yang namanya mengindikasikan layanan pengujian kecepatan milik entitas perbankan.")
print("Rekomendasi Operasional (bertahap): (1) periksa pola permintaan, konfigurasi, dan kemungkinan penggunaan sah;")
print("(2) verifikasi kondisi yang benar-benar membutuhkan pembatasan; (3) bila relevan, uji Response Rate Limiting")
print("(RRL) bertingkat berbasis tuple (subnet, domain, qtype) secara terbatas, dengan indikator dampak dan")
print("mekanisme pembatalan. Volume besar saja belum menunjukkan pembatasan dibutuhkan.")

# ---------------------------------------------------------------
# [Langkah 5] Pembersihan RAM
# Tujuan  : menghapus tabel kerja besar
# Alasan  : query mask besar tidak lagi dipakai di sel berikutnya
# ---------------------------------------------------------------
del queries_df_view, subnet_queries
_ = gc.collect()

## Cell 9: 7. Analisis Leksikal Bersih & Deteksi String Non-Standar
> Mengekstrak fitur karakter domain unik, mendeteksi pola escape octal (\xxx), dan mengonfirmasi cakupan TLD.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 7. ANALISIS LEKSIKAL DOMAIN & STRING ANOMALI ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Shannon entropy
# Tujuan  : menghitung keacakan karakter nama domain
# Alasan  : entropy tinggi menandai nama acak; nilai 0 untuk input kosong
# ---------------------------------------------------------------
def clean_shannon_entropy(s):
    """Hitung Shannon entropy string; 0.0 untuk input kosong atau non-string."""
    if not isinstance(s, str) or not s:
        return 0.0
    l = len(s)
    c = Counter(s)
    return -sum((cnt / l) * math.log2(cnt / l) for cnt in c.values())

# ---------------------------------------------------------------
# [Langkah 2] Tabel leksikal per domain unik
# Tujuan  : menyiapkan satu baris fitur per domain
# Alasan  : fitur dihitung per domain, bukan per paket, agar tidak dobel hitung
# ---------------------------------------------------------------
# Bangun tabel leksikal pada domain unik
lex_df = pd.DataFrame({'domain': pd.Series(clean_domain_series.unique())})

# ---------------------------------------------------------------
# [Langkah 3] Deteksi escape oktal
# Tujuan  : menandai nama berisi byte non-printable
# Alasan  : pola longgar dan ketat dibandingkan karena format exporter belum diverifikasi
# ---------------------------------------------------------------
# 1. Deteksi escape numerik tiga digit (format exporter belum diverifikasi)
octal_regex = re.compile(r'\\[0-9]{3}')
lex_df['has_numeric_escape'] = lex_df['domain'].str.contains(octal_regex, regex=True).astype('int8')

# ---------------------------------------------------------------
# [Langkah 4] Metrik karakter domain
# Tujuan  : menghitung panjang, label, digit, dan rasio digit
# Alasan  : operasi string vektor lebih cepat dari apply per baris
# ---------------------------------------------------------------
# 2. Metrik Karakter Domain
lex_df['qname_len'] = lex_df['domain'].str.len().astype('uint16')
lex_df['num_labels'] = (lex_df['domain'].str.count(r'\.') + 1).astype('uint8')
lex_df['longest_label_len'] = lex_df['domain'].apply(lambda x: max(map(len, x.split('.'))) if x else 0).astype('uint8')
lex_df['digit_count'] = lex_df['domain'].str.count(r'\d').astype('uint16')
lex_df['digit_ratio'] = (lex_df['digit_count'] / lex_df['qname_len']).astype('float32')
lex_df['entropy'] = lex_df['domain'].apply(clean_shannon_entropy).astype('float32')
lex_df['tld'] = lex_df['domain'].str.rsplit('.', n=1).str[-1].where(lex_df['domain'].str.contains('.', regex=False), 'none').astype('category')

# ---------------------------------------------------------------
# [Langkah 5] Ringkasan leksikal dan TLD
# Tujuan  : memvalidasi cakupan dataset lewat TLD
# Alasan  : proporsi TLD mengonfirmasi bahwa data memang zona .id
# ---------------------------------------------------------------
print(f"Total Domain Unik Dianalisis: {len(lex_df):,}")
print(f"  - Domain memuat escape numerik tiga digit : {lex_df['has_numeric_escape'].sum():,} domain ({lex_df['has_numeric_escape'].mean():.4%})")
octal_strict = re.compile(r'\\[0-7]{3}')
lex_df['has_numeric_escape_strict'] = lex_df['domain'].str.contains(octal_strict, regex=True).astype('int8')
print(f"  - Pola longgar [0-9]{{3}} = {int(lex_df['has_numeric_escape'].sum()):,} | ketat [0-7]{{3}} = {int(lex_df['has_numeric_escape_strict'].sum()):,} (oktal murni; format exporter belum diverifikasi)")
print(f"  - Catatan wire format: panjang di sini = string ter-escape (maks {int(lex_df['qname_len'].max())} karakter); batas DNS 255/63 octet dapat berbeda dari hitungan karakter ter-escape.\n")

print("[A] DISTRIBUSI STATISTIK METRIK LEKSIKAL:")
display(lex_df[['qname_len', 'num_labels', 'longest_label_len', 'digit_ratio', 'entropy']].describe().T)

print("\n[B] DISTRIBUSI TOP 5 TLD (Konfirmasi Cakupan Dataset):")
tld_dist = lex_df['tld'].value_counts().head(5)
display(pd.DataFrame({'TLD': tld_dist.index.astype(str), 'Total Domain Unik': tld_dist.values, 'Proporsi (%)': (tld_dist.values / len(lex_df)) * 100}))

## Cell 10: 8. Distribusi Persentil Empiris & Ambang Batas Data-Driven
> Menetapkan ambang batas objektif menggunakan persentil empiris (P50 s/d P99.9) untuk menangani distribusi berekor berat secara metodologis.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 8. DISTRIBUSI PERSENTIL EMPIRIS & AMBANG BATAS OBJEKTIF ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Tabel persentil empiris
# Tujuan  : menggambarkan distribusi fitur leksikal
# Alasan  : distribusi sangat miring sehingga persentil lebih informatif daripada mean
# ---------------------------------------------------------------
percentile_targets = ['qname_len', 'longest_label_len', 'entropy']
pct_table = lex_df[percentile_targets].quantile([0.5, 0.75, 0.90, 0.95, 0.99, 0.999]).T
pct_table.columns = ['Median (P50)', 'P75', 'P90', 'P95', 'P99', 'P99.9 (Extremes)']

print("[A] DISTRIBUSI PERSENTIL EMPIRIS FITUR LEKSIKAL:")
display(pct_table)

# ---------------------------------------------------------------
# [Langkah 2] Ambang batas data-driven
# Tujuan  : menetapkan P95, P99, dan P99.9 sebagai ambang anomali
# Alasan  : dipakai label proxy dan kandidat anomali; empiris dataset ini, belum tervalidasi eksternal
# ---------------------------------------------------------------
# Menetapkan ambang batas berbasis data
p999_len = float(lex_df['qname_len'].quantile(0.999))
p999_label = float(lex_df['longest_label_len'].quantile(0.999))
p99_entropy = float(lex_df['entropy'].quantile(0.99))
p95_len = float(lex_df['qname_len'].quantile(0.95))

print("\nAmbang Batas Objektif Berbasis Data (Data-Driven Thresholds):")
print(f"  - Ambang Panjang QNAME (P99.9)        : >= {p999_len:.0f} karakter")
print(f"  - Ambang Label Terpanjang (P99.9)     : >= {p999_label:.0f} karakter")
print(f"  - Ambang Shannon Entropy (P99.0)      : >= {p99_entropy:.2f}")
print("Catatan: persentil ini ambang empiris dataset ini saja, bukan ambang keamanan tervalidasi eksternal.")

## Cell 11: 9. Analisis Korelasi & Redundansi Fitur
> Menganalisis matriks korelasi rank Spearman antar variabel numerik dan menghitung contingency table antara Record Type dengan Response Code.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 9. KORELASI FITUR & TABEL KONTINJENSI ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Korelasi Spearman
# Tujuan  : mengukur ketergantungan antar fitur leksikal
# Alasan  : rank correlation tahan terhadap outlier dan distribusi miring
# ---------------------------------------------------------------
# Matriks Korelasi Spearman
num_cols = ['qname_len', 'num_labels', 'longest_label_len', 'digit_ratio', 'entropy']
spearman_matrix = lex_df[num_cols].corr(method='spearman')
print("[A] MATRIKS KORELASI SPEARMAN FITUR LEKSIKAL:")
display(spearman_matrix)

# ---------------------------------------------------------------
# [Langkah 2] Crosstab qtype vs rcode
# Tujuan  : melihat pemusatan NXDOMAIN per tipe query
# Alasan  : hanya respons karena RCODE bermakna pada respons, bukan query
# ---------------------------------------------------------------
# Crosstab: Top Record Type vs Response Code (hanya respons; RCODE bermakna pada respons)
resp_qtype = df.loc[r_mask, 'qtype_name']
resp_rcode = df.loc[r_mask, 'rcode']
print("\n[B] TABEL KONTINJENSI (%): TOP RECORD TYPE VS RCODE (DENOMINATOR: RESPONS):")
top5_qtypes = resp_qtype.value_counts().head(5).index
ct_qtype_rcode = pd.crosstab(
    resp_qtype[resp_qtype.isin(top5_qtypes)],
    resp_rcode,
    normalize='index'
) * 100
display(ct_qtype_rcode)

## Cell 12: 10. Deteksi Kandidat QNAME Anomali (Analisis PRSD & Evaluasi NXDOMAIN Host)
> Mengoreksi arah bukti DGA secara jujur (88,7% kandidat entropi tinggi memenuhi pola karakter yang diperiksa), mengelompokkannya per domain induk (registered domain) untuk menyaring kandidat pemeriksaan pola Pseudo-Random Subdomain (PRSD) / DNS Water Torture lebih lanjut, serta menghitung rasio NXDOMAIN host dengan denominator yang valid.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 10. DETEKSI KANDIDAT QNAME ANOMALI & AUDIT KARAKTERISTIK LEKSIKAL EKSTREM ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Kepatuhan charset RFC
# Tujuan  : menandai nama berisi karakter di luar LDH
# Alasan  : karakter asing menandai payload biner, escape, atau fuzzing
# ---------------------------------------------------------------
# 1. Pemetaan pola karakter terpilih (bukan pemeriksaan validitas DNS lengkap; DNS mengizinkan label biner)
foreign_chars_pattern = re.compile(r'[^a-z0-9\.\-_]')
lex_df['foreign_char_count'] = lex_df['domain'].apply(lambda x: len(foreign_chars_pattern.findall(x))).astype('uint16')
lex_df['is_selected_charset'] = (lex_df['foreign_char_count'] == 0).astype('int8')

# ---------------------------------------------------------------
# [Langkah 2] Seleksi kandidat entropi tinggi
# Tujuan  : mengambil domain unik yang entropinya ekstrem DAN namanya panjang
# Alasan  : entropi tinggi saja bisa muncul pada nama pendek yang sah, jadi dikombinasikan dengan panjang minimal P95
# ---------------------------------------------------------------
is_high_entropy = lex_df['entropy'] >= p99_entropy     # P99 entropi
is_long_name = lex_df['qname_len'] >= p95_len          # P95 panjang qname
candidate_high_entropy = lex_df[is_high_entropy & is_long_name].copy()
invalid_charset_pct = (candidate_high_entropy['is_selected_charset'] == 0).mean() * 100
valid_charset_pct = 100.0 - invalid_charset_pct

print(f"[A] PEMERIKSAAN SINTAKSIS KANDIDAT HIGH-ENTROPY ({len(candidate_high_entropy):,} domain):")
print(f"  - Memuat karakter di luar pola terpilih [a-z0-9.-_] : {invalid_charset_pct:.2f}% ({candidate_high_entropy['is_selected_charset'].eq(0).sum():,} domain)")
print(f"  - Memenuhi pola karakter terpilih (indikatif anomali) : {valid_charset_pct:.2f}% ({candidate_high_entropy['is_selected_charset'].eq(1).sum():,} domain)")

# 3. Analisis SLD Induk (2 Label Terakhir) pada Kandidat Alfanumerik Valid
# ---------------------------------------------------------------
# [Langkah 3] SLD induk dua label
# Tujuan  : melihat ke casar ke mana kandidat terkonsentrasi
# Alasan  : pengelompokan dua label terakhir adalah gambaran awal; registrable domain dihitung via PSL
# ---------------------------------------------------------------
valid_candidates = candidate_high_entropy[candidate_high_entropy['is_selected_charset'] == 1].copy()
valid_candidates['parent_domain'] = valid_candidates['domain'].apply(lambda x: '.'.join(x.split('.')[-2:]) if len(x.split('.')) >= 2 else x)

top_parent_domains = valid_candidates['parent_domain'].value_counts().head(5)
print("\n[B] KONSENTRASI SLD INDUK (2 LABEL TERAKHIR) PADA KANDIDAT ALFANUMERIK:")
display(pd.DataFrame({
    'Parent Domain': top_parent_domains.index,
    'Jumlah QNAME Kandidat': top_parent_domains.values,
    'Proporsi (%)': (top_parent_domains.values / len(valid_candidates)) * 100
}))
print("Catatan: pengelompokan SLD (2 label) dan registrable domain memakai Public Suffix List (snapshot")
print("bawaan publicsuffix2); jumlah registrable domain bukan jumlah registran - satu registran bisa")
print("memegang banyak domain. Konsentrasi pada SLD publik juga bukan bukti mandiri (dominan di zona .id).")

# ---------------------------------------------------------------
# [Langkah 4b] Sampel audit manual lintas kelompok
# Tujuan  : menyediakan contoh kandidat beserta alasan penandaan
# Alasan  : bentuk nama saja tidak cukup (contoh nyata: nama SRV Windows
#           terlihat anomali padahal sah); kolom penilaian manusia
#           (kemungkinan penggunaan sah, bukti tambahan) diisi manual
# ---------------------------------------------------------------
sampel_sah = valid_candidates.head(3).assign(
    alasan='memenuhi pola karakter terpilih')
sampel_luar = candidate_high_entropy[
    candidate_high_entropy['is_selected_charset'] == 0].head(3).assign(
    alasan='karakter di luar pola terpilih')
sampel_audit = pd.concat([sampel_sah, sampel_luar])[
    ['domain', 'alasan', 'qname_len', 'entropy']]
print("Sampel UNTUK AUDIT - belum dinilai; lengkapi manual: kemungkinan penggunaan sah, bukti tambahan yang diperlukan:")
display(sampel_audit)

# ---------------------------------------------------------------
# [Langkah 4] Registrable domain via PSL
# Tujuan  : menguji apakah kandidat terkonsentrasi pada sedikit pemilik zona
# Alasan  : PSL akurat untuk suffix majemuk; hasilnya jumlah domain, bukan jumlah pemilik
# ---------------------------------------------------------------
# 3b. Struktur domain registrabel (Public Suffix List)
psl = publicsuffix2.PublicSuffixList()
valid_candidates['registrable'] = valid_candidates['domain'].apply(lambda x: psl.get_sld(x, strict=False) or x)
reg_counts = valid_candidates['registrable'].value_counts()
n_reg = len(reg_counts)
n_valid_cand = len(valid_candidates)
reg_top1_share = (reg_counts.iloc[0] / n_valid_cand) * 100 if n_reg else 0.0
reg_top5_share = (reg_counts.head(5).sum() / n_valid_cand) * 100 if n_valid_cand else 0.0
nested_n = int((valid_candidates['domain'] != valid_candidates['registrable']).sum())
direct_n = n_valid_cand - nested_n

print(f"\n[B2] STRUKTUR DOMAIN REGISTRABEL (PSL): {n_reg:,} registrable domain berbeda dari {n_valid_cand:,} kandidat:")
print(f"  - Bersarang (subdomain di bawah registrable): {nested_n:,} | Kandidat = registrable itu sendiri: {direct_n:,}")
print(
    f"  - Top-5 registrable domain menampung "
    f"{reg_top5_share:.2f}% kandidat; "
    "kandidat tersebar pada banyak registrable domain. "
    "Jumlah pemilik domain tidak diketahui."
)
display(pd.DataFrame({
    'Registrable Domain (PSL)': reg_counts.head(5).index,
    'Jumlah QNAME Kandidat': reg_counts.head(5).values,
    'Proporsi (%)': (reg_counts.head(5).values / n_valid_cand) * 100
}))

# ---------------------------------------------------------------
# [Langkah 5] Konsentrasi sumber kandidat
# Tujuan  : membandingkan konsentrasi sumber kandidat vs trafik global
# Alasan  : top-5 sumber kandidat jauh lebih terkonsentrasi daripada global menandai sumber khusus
# ---------------------------------------------------------------
# 3c. Konsentrasi Sumber Query Kandidat (sumber khusus vs resolver umum)
cand_set = set(valid_candidates['domain'])
norm_qname_all = df['qname'].str.rstrip('.').str.lower()
cand_q_mask = norm_qname_all.isin(cand_set) & q_mask
cand_query_volume = int(cand_q_mask.sum())
cand_query_share = (cand_query_volume / total_q) * 100 if total_q else 0.0
cand_qname_counts = df.loc[cand_q_mask, 'qname'].str.rstrip('.').str.lower().value_counts()
once_seen_n = int((cand_qname_counts == 1).sum())

cand_src_counts = df.loc[cand_q_mask, 'src_ip'].value_counts()
cand_n_src = len(cand_src_counts)
cand_top5_share = (cand_src_counts.head(5).sum() / cand_src_counts.sum()) * 100 if cand_n_src else 0.0

global_src_counts = df.loc[q_mask, 'src_ip'].value_counts()
global_n_src = len(global_src_counts)
global_top5_share = (global_src_counts.head(5).sum() / global_src_counts.sum()) * 100 if global_n_src else 0.0

print(f"\n[B3] KONSENTRASI SUMBER QUERY KANDIDAT (VOLUME {cand_query_volume:,} query = {cand_query_share:.3f}% dari seluruh query):")
print(f"  - Kandidat : {cand_n_src:,} IP sumber unik, top-5 menyerap {cand_top5_share:.2f}% query kandidat")
print(f"  - Global   : {global_n_src:,} IP sumber unik, top-5 menyerap {global_top5_share:.2f}% query global")
print(f"  - Domain kandidat yang pernah di-query: {len(cand_qname_counts):,} | muncul sekali (single-query): {once_seen_n:,}")
sample_cand = valid_candidates['domain'].drop_duplicates().sample(min(10, n_valid_cand), random_state=RANDOM_SEED).tolist()
print("  - Sampel kandidat utk pemeriksaan manual (kemungkinan penggunaan sah):")
for d in sample_cand:
    print(f"      {d}")

# ---------------------------------------------------------------
# [Langkah 6] Kontribusi ke NXDOMAIN
# Tujuan  : mengukur porsi NXDOMAIN yang berasal dari kandidat
# Alasan  : denominator = seluruh NXDOMAIN agar tidak melebih-lebihkan peran kandidat
# ---------------------------------------------------------------
# 3d. Kontribusi Kandidat Anomali terhadap NXDOMAIN (denominator: seluruh NXDOMAIN)
nx_norm = df.loc[r_mask & (df['rcode'] == 3), ['dst_ip', 'qname']].copy()
nx_norm['clean_qname'] = nx_norm['qname'].str.rstrip('.').str.lower()
nx_norm['is_prsd'] = nx_norm['clean_qname'].isin(cand_set)
total_nx = int(len(nx_norm))
nx_prsd = int(nx_norm['is_prsd'].sum())
nx_prsd_share = (nx_prsd / total_nx) * 100 if total_nx else 0.0
per_host_prsd = nx_norm.groupby('dst_ip')['is_prsd'].mean() * 100  # porsi NX kandidat per host

nx_rate_global = (df.loc[r_mask, 'rcode'] == 3).mean() * 100 if total_r else 0.0
print('     Cakupan: kandidat entropi-panjang yang juga memenuhi pola karakter terpilih; '
      'bukan seluruh anomali leksikal.')

print("\nInsight Kritis (berbasis angka):")
print(f"  - {valid_charset_pct:.2f}% kandidat yang memenuhi pola tersebar di {n_reg:,} registrable domain (PSL; top-1 hanya {reg_top1_share:.2f}%) -> TIDAK terkonsentrasi pada segelintir registrable domain.")
print(f"  - Sumber query kandidat sangat terkonsentrasi ({cand_top5_share:.2f}% dari {cand_n_src:,} IP vs global {global_top5_share:.2f}%) -> pengiriman massal oleh sedikit sumber; kandidat ber-karakteristik leksikal ekstrem yang memerlukan investigasi lanjutan.")
print(f"  - Kandidat anomali hanya menyumbang {nx_prsd_share:.2f}% NXDOMAIN -> tingginya NXDOMAIN global ({nx_rate_global:.2f}%) didorong faktor lain di luar kandidat ini.")

# ---------------------------------------------------------------
# [Langkah 7] Audit host penerima NXDOMAIN
# Tujuan  : menghitung rasio NXDOMAIN per host penerima
# Alasan  : minimum respons diperlukan agar rasio per host bermakna
# ---------------------------------------------------------------
# 4. Evaluasi Target Host Penerima NXDOMAIN (Dengan Denominator Valid)
host_r_total = df.loc[r_mask, 'dst_ip'].value_counts()
host_nx_total = df.loc[r_mask & (df['rcode'] == 3), 'dst_ip'].value_counts()
host_norm_qname = df.loc[r_mask, 'qname'].str.rstrip('.').str.lower()
host_unique_q = host_norm_qname.groupby(df.loc[r_mask, 'dst_ip']).nunique()  # ternormalisasi 0x20

host_audit_df = pd.DataFrame({
    'Total Respons Diterima': host_r_total,
    'Total NXDOMAIN': host_nx_total,
    'QNAME Unik Diminta': host_unique_q
}).fillna({'Total NXDOMAIN': 0, 'QNAME Unik Diminta': 0})  # host tanpa NX tetap masuk audit (bukan dibuang)
host_audit_df['Total NXDOMAIN'] = host_audit_df['Total NXDOMAIN'].astype('uint32')
host_audit_df['QNAME Unik Diminta'] = host_audit_df['QNAME Unik Diminta'].astype('uint32')

MIN_RESP_HOST = 500  # minimum respons agar rasio per host bermakna
host_audit_df = host_audit_df[host_audit_df['Total Respons Diterima'] >= MIN_RESP_HOST].copy()
host_audit_df['NXDOMAIN Rate (%)'] = (host_audit_df['Total NXDOMAIN'] / host_audit_df['Total Respons Diterima']) * 100
host_audit_df = host_audit_df.sort_values(by='Total NXDOMAIN', ascending=False)

# ---------------------------------------------------------------
# [Langkah 8] Jembatan sumber-penerima
# Tujuan  : menguji overlap antara penerima NX ekstrem dan penerima kandidat anomali
# Alasan  : ambang 10 persen dipakai untuk memisahkan overlap kuat dari kebetulan kecil
# ---------------------------------------------------------------
# Jembatan sumber-penerima: porsi NX tiap host dari kandidat anomali
host_audit_df['Porsi NX Kandidat Anomali (%)'] = host_audit_df.index.map(per_host_prsd).fillna(0.0)
bridge_mask = host_audit_df['Porsi NX Kandidat Anomali (%)'] >= 10
n_bridge = int(bridge_mask.sum())
bridge_max = float(host_audit_df.loc[bridge_mask, 'Porsi NX Kandidat Anomali (%)'].max()) if n_bridge else 0.0
if n_bridge:
    bridge_sentence = f"ada {n_bridge} host prioritas dengan porsi NX kandidat anomali >= 10% (tertinggi {bridge_max:.1f}%)"
else:
    bridge_sentence = "tidak ada overlap kuat antara host NX ekstrem dan kandidat anomali pada data ini"

print(f"\n[C] PRIORITAS INVESTIGASI HOST PENERIMA NXDOMAIN (Minimal {MIN_RESP_HOST} Respons):")
display(host_audit_df.head(5))
print("    CATATAN: tabel ini menampilkan IP asli untuk keperluan operasional internal; masking IP/domain "
    "untuk perlindungan data dan privasi trafik jaringan.")

print("\nInterpretasi:")
print(f"  - Jembatan [B3]-[C]: {bridge_sentence}.")
print("  - Host ekstrem lainnya (NX Rate tinggi, porsi anomali ~0%) kemungkinan scanner, DNS reputation checker,")
print("    atau penerima query typo massal - dicatat sebagai prioritas investigasi lanjutan tanpa memvonis.")

host_nx_rate_top5_lo = float(host_audit_df.head(5)['NXDOMAIN Rate (%)'].min())
host_nx_rate_top5_hi = float(host_audit_df.head(5)['NXDOMAIN Rate (%)'].max())
del candidate_high_entropy, valid_candidates, host_audit_df, nx_norm, norm_qname_all, cand_q_mask
_ = gc.collect()



## Cell 13: 13. Ringkasan Eksekutif & Rekomendasi Operasional Dinamis
> Mencetak laporan eksekutif akhir yang seluruh angkanya ditarik langsung dari variabel hasil komputasi tanpa teks hardcoded.

In [ ]:
print("=" * 75)
print("=== 13. RINGKASAN EKSEKUTIF AKHIR (METODOLOGIS VALID) ===")
print("=" * 75)

# ---------------------------------------------------------------
# [Langkah 1] Agregat ringkasan
# Tujuan  : mengumpulkan angka-angka kunci untuk narasi akhir
# Alasan  : seluruh angka naratif ditarik dari variabel hasil komputasi, tanpa hardcode
# ---------------------------------------------------------------
# Konsentrasi Pareto Domain
domain_hit_counts = clean_domain_series.value_counts()
top_domain_name = domain_hit_counts.index[0]
top_domain_volume = domain_hit_counts.iloc[0]
top_domain_share = (top_domain_volume / len(df)) * 100

udp_tc_rate = (tc_on_udp / total_udp_r) * 100 if total_udp_r else 0.0
tcp_traffic_rate = (tcp_traffic_total / len(df)) * 100 if len(df) else 0.0
nx_rate_global = (df.loc[r_mask, 'rcode'] == 3).mean() * 100 if total_r else 0.0

# ---------------------------------------------------------------
# [Langkah 2] Cetak ringkasan eksekutif
# Tujuan  : menyusun narasi akhir berbasis angka
# Alasan  : teks f-string memastikan narasi selalu sinkron dengan hasil hitung
# ---------------------------------------------------------------
executive_summary = f"""
1. VANTAGE POINT & OPERASIONAL INFRASTRUKTUR:
   - Posisi Koleksi Data : Profil konsisten dengan sisi Authoritative Name Server untuk zona .id.
   - Perilaku Klien      : Sebanyak {rd_pct:.2f}% query menyalakan rd=1 (sisanya {100-rd_pct:.2f}% rd=0) dan {cd_pct:.2f}% menyalakan cd=1 (konsisten dengan perilaku validating resolver).
   - Karakter Jawaban    : Dari seluruh respon, {referral_share:.2%} KONSISTEN DENGAN referral (aa=0; proksi, authority section belum diperiksa) dan {nodata_share:.2%} KONSISTEN DENGAN NODATA (aa=1), berdasarkan flag dan counter (konfirmasi penuh butuh isi authority section, RFC 2308).
   - Efisiensi Protokol  : Terjadi pemotongan pada {udp_tc_rate:.2f}% respons UDP (tc=1), pangsa {top2_desc} di antara respons UDP yang terpotong (bukan rasio terpotong per tipe); kaitan kausal dengan ukuran payload/buffer EDNS masih perlu pengujian lanjutan, berdampingan dengan volume TCP {tcp_traffic_rate:.2f}%.

2. KONSENTRASI PARETO & DNS 0x20:
   - Reduksi Kardinalitas: Selisih {reduction_diff:,} variasi QNAME mentah konsisten dengan mitigasi DNS 0x20 bit-randomization (Dagon et al., 2008). Kardinalitas mentah menggandakan hitungan domain riil ~2x.
   - Dominasi Domain     : Domain '{top_domain_name}' menyerap {top_domain_share:.2f}% dari seluruh log ({top_domain_volume:,} paket).
   - Konsentrasi Subnet  : Subnet {top_subnet_name} menyumbang {top_subnet_share:.2f}% query global dan mencakup {target_subnet_share:.2f}% kueri domain '{target_domain}'.
   - Rekomendasi Beban   : Bertahap - periksa pola permintaan dan penggunaan sah, verifikasi kebutuhan pembatasan, lalu uji RRL bertingkat (subnet, domain, qtype) terbatas dengan indikator dampak dan pembatalan.

3. TEMUAN ANOMALI LEKSIKAL:
   - Terdeteksi {lex_df['has_numeric_escape'].sum():,} domain dengan escape numerik tiga digit (makna bergantung format exporter; belum dibuktikan payload atau serangan).
   - Sebanyak {valid_charset_pct:.1f}% kandidat entropi tinggi yang memenuhi pola karakter terpilih tersebar di {n_reg:,} registrable domain berbeda (PSL; top-1 {reg_top1_share:.2f}%), tetapi sumbernya terkonsentrasi: {cand_n_src:,} IP unik dengan top-5 menyerap {cand_top5_share:.2f}% query kandidat (global: {global_top5_share:.2f}%), konsisten dengan pengiriman subdomain acak massal dari sedikit sumber - kandidat QNAME ber-karakteristik leksikal ekstrem yang memerlukan investigasi lanjutan.
   - Kontribusi kandidat anomali hanya {nx_prsd_share:.2f}% dari {total_nx:,} NXDOMAIN (cakupan: kandidat entropi-panjang berpola karakter terpilih);
   - Ambang anomali data-driven (P99.9) menetapkan panjang domain >= {p999_len:.0f} karakter dan label >= {p999_label:.0f} karakter.

4. KETERBATASAN (MASUKAN UNTUK LAPORAN):
   - Pengelompokan registrable memakai Public Suffix List (snapshot bawaan publicsuffix2); jumlah registrable domain bukan jumlah registran; referral/NODATA dipisah via bit aa (bukan isi authority section, RFC 2308).
   - df.loc[...] menghasilkan copy (bukan view): klaim efisiensi memori perlu pengukuran langsung; satuan ukuran = GiB (biner).
   - Pembatasan thread + gc.collect() = mitigasi risiko OOM, bukan jaminan.
   - Dugaan anonimisasi IP membatasi interpretasi prefix sebagai organisasi atau resolver nyata.
   - Belum dilakukan: laju kandidat per interval waktu, audit manual contoh QNAME masif.
   - Analisis leksikal dan deteksi PRSD disajikan secara deterministik berbasis PSL dan ambang batas data-driven (P99.9).
"""

print(executive_summary)


## Cell 14: 14. Sintesis Operasional untuk Tim NOC (Beban, Konsentrasi, Respons, Prioritas)

> Pertanyaan bisnis: *pola trafik dan respons DNS mana yang perlu diprioritaskan tim operasi untuk pemeriksaan lebih lanjut, berdasarkan besarnya beban, frekuensi masalah, dan konsentrasinya?*
> Setiap temuan ditutup dengan keputusan: tindakan yang direkomendasikan + indikator tindak lanjut. Batas interpretasi: NXDOMAIN bukan otomatis gangguan/serangan; volume tinggi bukan otomatis berbahaya; cakupan kesimpulan dibatasi pada periode pengamatan.

In [ ]:
from IPython.display import display
print("=" * 75)
print("=== 14. SINTESIS OPERASIONAL: BEBAN, KONSENTRASI, RESPON & PRIORITAS TINDAKAN ===")
print("=" * 75)

os.makedirs('../output', exist_ok=True)  # folder tujuan berkas gambar visualisasi

# Tema BI (matplotlib): palet terbatas, ink minim, judul takeaway
PAL = {'base': '#1F3A5F', 'muted': '#B8C2CC', 'accent': '#E4572E', 'ok': '#2A9D8F', 'ink': '#2B2D42', 'bg': '#FFFFFF'}
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110, 'figure.facecolor': PAL['bg'], 'axes.facecolor': PAL['bg'],
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.edgecolor': PAL['muted'], 'axes.labelcolor': PAL['ink'], 'axes.titlecolor': PAL['ink'],
    'xtick.color': PAL['ink'], 'ytick.color': PAL['ink'],
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#EEF1F4', 'grid.linewidth': 0.7, 'axes.axisbelow': True,
})
fmt_hm = mdates.DateFormatter('%H:%M')

# ---------------------------------------------------------------
# [Langkah 12] Label anonim subnet untuk visualisasi ringkas & privasi
# Tujuan  : mengganti prefix asli dengan S1, S2, dst pada semua visual
# Alasan  : abstraksi prefix untuk penyajian laporan dan privasi jaringan;
#           masking IP/domain menjaga keamanan infrastruktur
# ---------------------------------------------------------------
anon_map = {p: f'S{i + 1}' for i, p in enumerate(prefix_df['Subnet Prefix'])}

def anon(p):
    """Kembalikan label S# unik untuk prefix; tidak pernah memunculkan nilai asli."""
    if p not in anon_map:
        anon_map[p] = f'S{len(anon_map) + 1}'
    return anon_map[p]

# ---------------------------------------------------------------
# [Langkah 13] Audit duplikasi
# Tujuan  : melengkapi kelayakan data dengan cek duplikat
# Alasan  : duplikat kunci transaksi hanya indikasi karena dns_id
#           boleh berulang antar klien
# ---------------------------------------------------------------
dup_exact = int(df.duplicated().sum())
dup_key = int(df.duplicated(subset=['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'dns_id', 'qname', 'qr']).sum())
print(f"[A] KELAYAKAN DATA: baris duplikat persis = {dup_exact:,} | duplikat kunci transaksi = {dup_key:,} | cakupan {n_min_obs} menit penuh")
print("    Catatan: dns_id dapat berulang sah antar klien; angka kunci hanya indikasi, bukan bukti duplikasi.")

# ---------------------------------------------------------------
# [Langkah 14] Agregasi beban per menit (window 29 menit kanonik)
# Tujuan  : menghitung paket per menit dan pangsa prefix teratas
# Alasan  : pembilang dan denominator memakai 29 interval kanonik
#           yang sama; window menit parsial tidak dipakai
# ---------------------------------------------------------------
row_minute = df['ts_iso'].dt.floor('min')
in_canonical = row_minute.isin(steady_state_temporal.index)
q_work = df.loc[q_mask & in_canonical, ['ts_iso', 'src_ip']].copy()
q_work['prefix'] = q_work['src_ip'].map(prefix_map)
q_work['minute'] = q_work['ts_iso'].dt.floor('min')
q_work = q_work[q_work['minute'].isin(steady_state_temporal.index)]  # 29 menit kanonik
per_min_prefix = q_work.groupby([pd.Grouper(key='ts_iso', freq='1min'), 'prefix']).size()
per_min_total = q_work.groupby(pd.Grouper(key='ts_iso', freq='1min')).size()
top_prefix_per_min = per_min_prefix.xs(top_subnet_name, level='prefix').reindex(per_min_total.index).fillna(0)
top_share_fixed = top_prefix_per_min / per_min_total * 100
peak_minute = steady_state_temporal['packets'].idxmax()
median_lain = steady_state_temporal.loc[
    steady_state_temporal.index != peak_minute, 'packets'].median()  # median tanpa menit puncak
peak_top_prefix = per_min_prefix.loc[peak_minute].idxmax() if peak_minute in per_min_prefix.index.get_level_values(0) else '-'
print(f"[B] BEBAN: puncak {steady_state_temporal['packets'].max():,} paket/menit pada {peak_minute:%H:%M} UTC; kontributor teratas menit tsb: {anon(peak_top_prefix)}")
print(f"    Pangsa {top_subnet_name} per menit (prefix sama sepanjang waktu): min {top_share_fixed.min():.1f}% | median {top_share_fixed.median():.1f}% | maks {top_share_fixed.max():.1f}% | menit dgn pangsa >50%: {int((top_share_fixed > 50).sum())} dari {len(top_share_fixed)}")

# ---------------------------------------------------------------
# [Langkah 15] Agregasi NX dan truncation UDP per menit
# Tujuan  : derat temporal dua metrik masalah
# Alasan  : reindex ke 29 menit kanonik; denominator tc = respons UDP
# ---------------------------------------------------------------
# Agregasi per menit dari modul perbaikan: bin kanonik sama; bin kosong volume 0, rasio NaN
resp_min = temporal[['nx', 'responses', 'nx_rate']].rename(columns={'responses': 'resp'})
trunc_min = temporal[['udp_resp', 'udp_tc', 'tc_ratio']]
tc_ratio_avg = (100 * trunc_min['udp_tc'].sum() / trunc_min['udp_resp'].sum()
                if trunc_min['udp_resp'].sum() > 0 else float('nan'))
nx_every = 100 / nx_rate_global if nx_rate_global else 0
tc_every = 100 / tc_ratio_avg if tc_ratio_avg else 0

# KPI card: lima angka kunci di baris teratas
s1_share_window = q_work['prefix'].eq(top_subnet_name).mean() * 100  # pangsa S1 pada 29 menit kanonik
kpi = [
    ('Pesan DNS', f'{len(df) / 1e6:.1f}'.replace('.', ',') + ' jt', 'rekaman penuh'),
    ('Pesan / detik', f'{len(df) / (t_max - t_min).total_seconds():,.0f}', 'rekaman penuh'),
    ('NXDOMAIN', f'{nx_rate_global:.2f}%', 'rekaman penuh'),
    ('Truncation UDP', f'{tc_ratio_avg:.2f}%', '29 menit kanonik'),
    ('Pangsa S1', f'{s1_share_window:.1f}%', '29 menit kanonik'),
]
fig, ax = plt.subplots(figsize=(10, 1.4))
ax.axis('off')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
for i, (t, v, scope) in enumerate(kpi):
    x0 = i / 5
    ax.add_patch(plt.Rectangle((x0 + 0.008, 0.04), 0.184, 0.92, transform=ax.transAxes,
                               facecolor='#F4F6F8', edgecolor=PAL['muted'], linewidth=0.8))
    ax.text(x0 + 0.1, 0.74, t, transform=ax.transAxes, ha='center', fontsize=9, color=PAL['ink'])
    ax.text(x0 + 0.1, 0.38, v, transform=ax.transAxes, ha='center', fontsize=13, fontweight='bold', color=PAL['base'])
    ax.text(x0 + 0.1, 0.14, scope, transform=ax.transAxes, ha='center', fontsize=7.5, color='#5B6770')
fig.savefig('../output/01_kpi_card.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)

# Grafik 1: timeline bertumpuk (beban / NX rate / tc ratio) - satu sumbu x, tanpa dual axis
nx_peak_minute = resp_min['nx_rate'].idxmax()
fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(10, 6.4), sharex=True,
                                 gridspec_kw={'height_ratios': [1.4, 1, 1], 'hspace': 0.2})
a1.fill_between(steady_state_temporal.index, steady_state_temporal['packets'] / 1000, color=PAL['base'], alpha=0.22, linewidth=0)
a1.plot(steady_state_temporal.index, steady_state_temporal['packets'] / 1000, color=PAL['base'], linewidth=1.6)
a1.axhline(median_lain / 1000, color=PAL['muted'], linewidth=1, linestyle='--')
a1.set_ylabel('Paket/menit (ribu)')
a1.set_title(f'Beban puncak {steady_state_temporal["packets"].max():,} paket/menit @ {peak_minute:%H:%M} UTC', loc='left')
a2.plot(resp_min.index, resp_min['nx_rate'], color=PAL['accent'], linewidth=1.6, marker='o', markersize=3)
a2.axhline(nx_rate_global, color=PAL['muted'], linewidth=1, linestyle='--')  # rasio keseluruhan periode
a2.set_ylabel('NX rate (%)')
a2.set_title(f'Puncak NXDOMAIN {resp_min["nx_rate"].max():.1f}% @ {nx_peak_minute:%H:%M} UTC', loc='left')
a2.annotate(f'{resp_min["nx_rate"].max():.1f}% @ {nx_peak_minute:%H:%M}', xy=(nx_peak_minute, resp_min.loc[nx_peak_minute, 'nx_rate']),
            xytext=(-120, -16), textcoords='offset points', fontsize=9, color=PAL['accent'],
            arrowprops=dict(arrowstyle='->', color=PAL['accent'], lw=1))
a3.plot(trunc_min.index, trunc_min['tc_ratio'].astype('float'), color=PAL['ok'], linewidth=1.6, marker='o', markersize=3)
a3.axhline(tc_ratio_avg, color=PAL['muted'], linewidth=1, linestyle='--')
a3.set_ylabel('tc ratio (%)')
tc_peak_minute = trunc_min['tc_ratio'].astype('float').idxmax()
tc_peak_val = float(trunc_min.loc[tc_peak_minute, 'tc_ratio'])
if tc_peak_minute == nx_peak_minute:
    a3.set_title(f'Truncation UDP meningkat pada {tc_peak_minute:%H:%M} UTC, bersamaan dengan puncak NXDOMAIN', loc='left')
else:
    a3.set_title(f'Truncation UDP puncak {tc_peak_val:.1f}% @ {tc_peak_minute:%H:%M} UTC (puncak NXDOMAIN @ {nx_peak_minute:%H:%M})', loc='left')
a3.xaxis.set_major_formatter(fmt_hm)
fig.align_ylabels()
fig.savefig('../output/02_timeline_beban_nx_tc.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
print("   Bacaan: panel atas = beban; tengah = porsi respons NXDOMAIN (nama tidak ditemukan); bawah = porsi jawaban UDP terpotong. Garis putus: beban = median menit lain (tanpa puncak); NX = rasio keseluruhan rekaman; truncation = rasio agregat 29 menit.")

# Grafik 2: Pareto sumber (label anonim S1..S10) - batang & kumulatif pada satu sumbu 0-100%
pareto = prefix_df.copy()
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.bar([anon(p) for p in pareto['Subnet Prefix']], pareto['Share (%)'], color=PAL['base'], label='Pangsa query')
ax.plot([anon(p) for p in pareto['Subnet Prefix']], pareto['Kumulatif (%)'], color=PAL['accent'], marker='o', markersize=4, linewidth=1.5, label='Kumulatif')
ax.set_ylim(0, 105)
ax.set_ylabel('% dari seluruh query')
ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0.04, 0.98), fontsize=9)
for i, v in enumerate(pareto['Share (%)']):
    ax.text(i, v + 2, f'{v:.1f}', ha='center', fontsize=8.5)
ax.set_title(f'S1 sendiri menyerap {pareto["Share (%)"].iloc[0]:.1f}% query; {len(pareto)} subnet teratas = {pareto["Kumulatif (%)"].iloc[-1]:.1f}%', loc='left')
fig.savefig('../output/03_pareto_sumber.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
print("   Bacaan: S1..S10 = urutan subnet sumber dengan query terbanyak (abstraksi anonim untuk privasi; cakupan rekaman penuh).")

# Grafik 3: komposisi QTYPE (query) - top 5 + Lainnya
qtype_vc = df.loc[q_mask, 'qtype_name'].value_counts()
top5q = qtype_vc.head(5)
qt_labels = [str(x) for x in top5q.index] + ['Lainnya']
qt_vals = list(top5q.values) + [int(qtype_vc.iloc[5:].sum())]
fig, ax = plt.subplots(figsize=(10, 3.0))
bars = ax.barh(list(reversed(qt_labels)), [v / 1000 for v in reversed(qt_vals)], color=PAL['base'])
for b, v in zip(bars, list(reversed(qt_vals))):
    ax.text(b.get_width() * 1.01, b.get_y() + b.get_height() / 2, f'{v / total_q * 100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Query (ribu)')
ax.set_xlim(0, max(qt_vals) / 1000 * 1.16)
ax.set_title(f'Komposisi QTYPE - {qt_labels[0]} terbanyak ({qt_vals[0] / total_q * 100:.1f}% dari {total_q:,} query)', loc='left')
fig.savefig('../output/04_komposisi_qtype.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
print("   Bacaan: A/AAAA = pencarian alamat IP; NS/DS = delegasi & DNSSEC; MX = email; TXT = teks (SPF/DKIM/DMARC).")

# Grafik 4: komposisi RCODE (respons) - top 4 + Lainnya
rc_vc = df.loc[r_mask, 'rcode'].value_counts()
top4r = rc_vc.head(4)
rc_labels = [rcode_map.get(int(k), f'CODE_{int(k)}') for k in top4r.index] + ['Lainnya']
rc_vals = list(top4r.values) + [int(rc_vc.iloc[4:].sum())]
fig, ax = plt.subplots(figsize=(10, 2.8))
bars = ax.barh(list(reversed(rc_labels)), [v / 1000 for v in reversed(rc_vals)], color=PAL['ok'])
for b, v in zip(bars, list(reversed(rc_vals))):
    _pct = v / total_r * 100
    ax.text(b.get_width() * 1.01, b.get_y() + b.get_height() / 2,
            f'<0.01% ({v:,})' if _pct < 0.005 else f'{_pct:.2f}%', va='center', fontsize=9)
ax.set_xlabel('Respons (ribu)')
ax.set_xlim(0, max(rc_vals) / 1000 * 1.16)
ax.set_title(f'Komposisi RCODE pada respons - {rc_labels[0]} dominan ({rc_vals[0] / total_r * 100:.1f}%)', loc='left')
fig.savefig('../output/05_komposisi_rcode.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
print("   Bacaan: NOERROR = tidak ada kode kesalahan; NXDOMAIN = nama tidak ada; SERVFAIL = server gagal; REFUSED = ditolak.")

# Grafik 5: heatmap qtype x rcode (% per baris, respons)
mat = ct_qtype_rcode
fig, ax = plt.subplots(figsize=(10, 3.2))
im = ax.imshow(mat.values, cmap='Blues', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(mat.shape[1]))
ax.set_xticklabels([rcode_map.get(int(c), str(c)) for c in mat.columns], fontsize=9)
ax.set_yticks(range(mat.shape[0]))
ax.set_yticklabels([str(x) for x in mat.index], fontsize=9)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.values[i, j]
        if v >= 1:
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color='white' if v > 60 else PAL['ink'])
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.85, label='% respons')
if 3 in mat.columns:
    _qt_nx = mat[3].idxmax()
    ax.set_title(f'qtype x rcode (% per baris) - proporsi NXDOMAIN tertinggi pada {_qt_nx} ({mat.loc[_qt_nx, 3]:.1f}% respons tipe tsb; dari {mat.shape[0]} tipe yang ditampilkan)', loc='left')
else:
    ax.set_title('qtype x rcode (% per baris) - proporsi rcode berbeda antar tipe', loc='left')
fig.savefig('../output/06_heatmap_qtype_rcode.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)

# Grafik 6: truncation per QTYPE (rate, bukan hanya pangsa; minimal 500 respons UDP)
udp_mask = r_mask & (df['proto'] == 'udp')
tq_resp = df.loc[udp_mask, 'qtype_name'].value_counts()
tq_tc = df.loc[udp_mask & (df['tc'] == 1), 'qtype_name'].value_counts()
tq = pd.DataFrame({'resp': tq_resp, 'tc': tq_tc}).fillna(0)
MIN_RESPONSES = 500  # minimum respons agar rate per qtype bermakna
tq = tq[tq['resp'] >= MIN_RESPONSES]
tq['rate'] = tq['tc'] / tq['resp'] * 100
tq = tq.sort_values('rate', ascending=False).head(8)
fig, ax = plt.subplots(figsize=(10, 3.2))
bars = ax.barh([str(x) for x in tq.index][::-1], tq['rate'][::-1], color=PAL['ok'])
for b, v, n_resp in zip(bars, tq['rate'][::-1], tq['resp'][::-1]):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2, f'{v:.1f}% ({int(n_resp):,} resp)', va='center', fontsize=9)
ax.set_xlabel('Rasio tc=1 terhadap respons UDP (%)')
ax.set_xlim(0, tq['rate'].max() * 1.32)
ax.set_title(f'Truncation per QTYPE - {tq.index[0]} paling sering terpotong ({tq["rate"].iloc[0]:.1f}%)', loc='left')
fig.savefig('../output/07_truncation_per_qtype.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
print(f"   Bacaan: rasio = jawaban UDP terpotong / seluruh respons UDP tipe tsb (minimal {MIN_RESPONSES} respons).")

# Grafik 7: distribusi panjang qname (log) + ambang empiris P95/P99.9
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.hist(lex_df['qname_len'], bins=60, color=PAL['base'], alpha=0.85)
ax.set_yscale('log')
ax.axvline(p95_len, color=PAL['ok'], linewidth=1.2, linestyle='--')
ax.axvline(p999_len, color=PAL['accent'], linewidth=1.2, linestyle='--')
ymin_log, ymax_log = ax.get_ylim()
y_lbl = ymin_log * (ymax_log / ymin_log) ** 0.72  # posisi 72% tinggi panel pada skala log
ax.text(p95_len * 1.02, y_lbl, f'P95 = {p95_len:.0f}', fontsize=8.5, color=PAL['ok'])
ax.text(p999_len * 1.02, y_lbl, f'P99.9 = {p999_len:.0f}', fontsize=8.5, color=PAL['accent'])
ax.set_xlabel('Panjang qname (karakter, setelah escape)')
ax.set_ylabel('Jumlah domain (log)')
ax.set_title(f'Panjang qname sangat miring - ekor panjang melewati P99.9 ({p999_len:.0f} karakter)', loc='left')
fig.savefig('../output/08_hist_panjang_qname.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
print("   Bacaan: sumbu y log - setiap tingkat = 10x lipat; garis hijau/merah = ambang empiris P95/P99.9 dataset ini.")

# Prioritas subnet sumber: agregasi dua daftar (beban & masalah)
r_view = df.loc[r_mask & in_canonical, ['dst_ip', 'rcode']].copy()
r_view['prefix'] = r_view['dst_ip'].map(prefix_map)
unmapped = r_view.loc[r_view['prefix'].isna(), 'dst_ip'].unique()
if len(unmapped):
    extra_map = {ip: fast_prefix(ip) for ip in unmapped}
    r_view['prefix'] = r_view['prefix'].fillna(r_view['dst_ip'].map(extra_map))
prio = priority_sources(q_work[['prefix', 'minute']], r_view[['prefix', 'rcode']])
print(f"  Cakupan peta: prefix hanya di respons (query=0): "
      f"{int(((prio['query'] == 0) & (prio['respons'] > 0)).sum())} | hanya di query (respons=0): "
      f"{int(((prio['respons'] == 0) & (prio['query'] > 0)).sum())}")

# I1: prioritas beban (urut volume query)
f1 = prio[prio['query'] > 0].sort_values('query', ascending=False).head(10).copy()
alasan_f1 = []
for rank, (name, row) in enumerate(f1.iterrows()):
    reasons = []
    if rank < 3:
        reasons.append('volume terbesar')
    if row['menit_aktif'] >= n_min_obs - 3:
        reasons.append('aktif hampir sepanjang periode')
    alasan_f1.append('; '.join(reasons) if reasons else 'pemantauan rutin')
f1['Alasan Prioritas'] = alasan_f1
f1_disp = f1.reset_index()
f1_disp['Subnet Sumber'] = f1_disp['prefix'].map(anon)
print("[I1] PRIORITAS BEBAN (urut volume query; pembilang & denominator menit memakai interval yang sama):")
display(f1_disp.rename(columns={'query': 'Query', 'share_pct': 'Share Query (%)', 'menit_aktif': f'Menit Aktif (dari {n_min_obs})'})[
    ['Subnet Sumber', 'Query', 'Share Query (%)', f'Menit Aktif (dari {n_min_obs})', 'Alasan Prioritas']])

# I2: prioritas respons bermasalah (urut NXDOMAIN, syarat volume respons minimal)
f2 = prio[prio['respons'] >= 500].sort_values('nxdomain', ascending=False).head(10).copy()
alasan_f2 = []
for rank, (name, row) in enumerate(f2.iterrows()):
    reasons = []
    if rank < 3:
        reasons.append('volume NX terbesar')
    if row['nx_rate_pct'] >= 25:
        reasons.append('NX rate tinggi')
    alasan_f2.append('; '.join(reasons) if reasons else 'rasio perlu dipantau')
f2['Alasan Prioritas'] = alasan_f2
f2_disp = f2.reset_index()
f2_disp['Subnet Sumber'] = f2_disp['prefix'].map(anon)
print("[I2] PRIORITAS RESPONS BERMASALAH (urut NXDOMAIN; syarat respons >= 500; host ekstrem individual: tabel [C] sel 10):")
display(f2_disp.rename(columns={'nxdomain': 'NXDOMAIN', 'nx_rate_pct': 'NX Rate (%)', 'respons': 'Respons Diterima'})[
    ['Subnet Sumber', 'Respons Diterima', 'NXDOMAIN', 'NX Rate (%)', 'Alasan Prioritas']])
print("    Catatan: daftar beban dan daftar masalah tidak selalu beririsan; tabel memakai label anonim S# (pemetaan asli internal).")

# Grafik 8: bubble prioritas - gabungan beban teratas dan NX teratas; label terpisah dari kluster
by_query = prio.sort_values('query', ascending=False).head(10)
by_nx = prio.sort_values('nxdomain', ascending=False).head(10)
bub = pd.concat([by_query, by_nx]).reset_index().drop_duplicates(subset=['prefix'])
extend_anon_map(anon_map, bub['prefix'])  # label baru selalu unik terhadap label lama
assert len(set(anon_map.values())) == len(anon_map)
bub_plot = bub.loc[(bub['query'] > 0) & bub['nx_rate_pct'].notna()].copy()
bub_lain = bub.loc[~bub.index.isin(bub_plot.index)]
if len(bub_lain):
    _lain = bub_lain.reset_index()
    _lain['Subnet Sumber'] = _lain['prefix'].map(anon)
    print('   Sumber tidak terplot pada bidang log (query=0 atau tanpa respons):')
    display(_lain[['Subnet Sumber', 'query', 'respons', 'nxdomain', 'nx_rate_pct']])
bub = bub_plot  # sel grafik berikut memakai subset terplot; sumber lain tetap tampil sebagai tabel
lab_mask = [(i < 4) or (r['nx_rate_pct'] >= 25)  # label: 4 beban teratas + semua sumber NX rate tinggi
            for i, (_, r) in enumerate(bub.iterrows())]
n_lab = int(sum(lab_mask))
lab_idx_sorted = sorted([i for i, m in enumerate(lab_mask) if m],
                        key=lambda i: (bub.iloc[i]['nx_rate_pct'], bub.iloc[i]['query']))
off_by_idx = {}  # titik rendah berlabel di atas agar tidak terpotong; sisanya selang-seling
_k = 0
for _i in lab_idx_sorted:
    if bub.iloc[_i]['nx_rate_pct'] < 8:
        off_by_idx[_i] = (6, 6)
    else:
        off_by_idx[_i] = [(6, 6), (6, -16), (-46, 6), (-46, -16)][_k % 4]
        _k += 1
# pasangan label yang berdekatan (y dan x mirip) dibelokkan ke kiri agar tidak bertumpuk
for _a in range(len(lab_idx_sorted)):
    for _b in range(_a + 1, len(lab_idx_sorted)):
        _ia, _ib = lab_idx_sorted[_a], lab_idx_sorted[_b]
        _ya, _yb = bub.iloc[_ia]['nx_rate_pct'], bub.iloc[_ib]['nx_rate_pct']
        _xa, _xb = bub.iloc[_ia]['query'], bub.iloc[_ib]['query']
        if abs(_ya - _yb) < 4 and 0.8 < _xb / _xa < 1.25 and off_by_idx[_ia] == off_by_idx[_ib]:
            off_by_idx[_ib] = (-40, 6)
            break
fig, ax = plt.subplots(figsize=(10, 3.8))
for i, (_, r) in enumerate(bub.iterrows()):
    color = PAL['accent'] if r['nx_rate_pct'] >= 25 else PAL['base']
    ax.scatter(r['query'], r['nx_rate_pct'], s=30 + r['respons'] / 3000, color=color, alpha=0.75,
               edgecolor='white', linewidth=0.8, zorder=3)
    if lab_mask[i]:  # label hanya sumber yang terpisah secara visual
        ax.annotate(anon(r['prefix']), (r['query'], r['nx_rate_pct']),
                    xytext=off_by_idx[i], textcoords='offset points', fontsize=8.5)
ax.set_xscale('log')
ax.set_xlabel('Volume query (skala log)')
ax.set_ylabel('NX rate (%)')
ax.set_ylim(-max(bub['nx_rate_pct']) * 0.06, max(bub['nx_rate_pct']) * 1.35)  # ruang bawah agar gelembung besar tidak terpotong
unlab = bub[~pd.Series(lab_mask, index=bub.index)]
if len(unlab) >= 3:
    med_u = unlab['nx_rate_pct'].median()
    cx_u = unlab['query'].median()
    ax.annotate(f'{len(unlab)} sumber lain: NX rate {unlab["nx_rate_pct"].min():.0f}-'
                f'{unlab["nx_rate_pct"].max():.0f}%',
                xy=(cx_u, med_u), xytext=(cx_u, 42), textcoords='data', ha='center',
                fontsize=9, color=PAL['ink'],
                arrowprops=dict(arrowstyle='->', color=PAL['muted'], lw=1))
ax.text(0.45, 0.05, 'Ukuran titik = volume respons sumber', transform=ax.transAxes,
        ha='center', fontsize=8.5, color='#5B6770')
ax.set_title('Volume query dan proporsi NXDOMAIN per sumber; titik merah = NX rate >= 25%', loc='left')
fig.savefig('../output/09_bubble_prioritas.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)

print("   Bacaan: ukuran gelembung = volume respons sumber; titik merah = NX rate >= 25%; label = subnet anonim. "
      "Tabel [I1]/[I2] adalah versi ringkas untuk laporan; grafik ini lampiran.")

# Grafik 9: small multiples burst - tiga sumber teratas, sumbu y sama, puncak per sumber
top3p = list(prio.sort_values('query', ascending=False).index[:3])  # benar-benar tiga teratas
pivot3 = per_min_prefix.unstack('prefix').reindex(steady_state_temporal.index)[top3p].fillna(0)
fig, axes = plt.subplots(3, 1, figsize=(10, 4.8), sharex=True, sharey=True)
for _k, (ax_i, p) in enumerate(zip(axes, top3p)):
    ser = pivot3[p]
    pk_minute, pk_val = ser.idxmax(), ser.max()
    ax_i.plot(pivot3.index, ser / 1000, color=PAL['base'], linewidth=1.5)
    ax_i.fill_between(pivot3.index, ser / 1000, color=PAL['base'], alpha=0.18, linewidth=0)
    share_p = prio.loc[p, 'share_pct']
    ax_i.set_ylabel(f'{anon(p)} ({share_p:.1f}%)\n(ribu/menit)', fontsize=9)
    ax_i.axvline(peak_minute, color=PAL['muted'], linewidth=1, linestyle=':')  # puncak global
    ax_i.plot([pk_minute], [pk_val / 1000], 'o', color=PAL['accent'], markersize=5)
    ax_i.annotate(f'puncak {pk_minute:%H:%M}', xy=(pk_minute, pk_val / 1000),
                  xytext=(6, -16) if _k == 0 else (6, 6), textcoords='offset points', fontsize=8.5, color=PAL['accent'])
axes[0].set_title(f'Trafik query per menit tiga sumber teratas - puncak global {peak_minute:%H:%M} UTC (garis titik-titik)', loc='left')
axes[-1].xaxis.set_major_formatter(fmt_hm)
fig.align_ylabels()
fig.savefig('../output/10_small_multiples_burst.png', bbox_inches='tight')  # simpan versi final untuk paket
display(fig)
plt.close(fig)
s_share = [prio.loc[p, 'share_pct'] for p in top3p]
print("   Bacaan: skala y sama agar perbandingan jujur; S2 dan S3 tampak datar karena pangsanya hanya "
      f"{s_share[1]:.1f}% dan {s_share[2]:.1f}% dari total query. Titik oranye = menit puncak masing-masing sumber.")
del q_work, r_view

# Profil menit puncak beban - deskriptif, tanpa menyimpulkan penyebab
m_mask = (df['ts_iso'] >= peak_minute) & (df['ts_iso'] < peak_minute + pd.Timedelta('1min'))
m_df = df.loc[m_mask]
m_q = int((m_df['qr'] == 0).sum())
m_r = int((m_df['qr'] == 1).sum())
m_nx = int(((m_df['qr'] == 1) & (m_df['rcode'] == 3)).sum())
m_nx_rate = m_nx / m_r * 100 if m_r else 0.0
m_top_qtype = m_df.loc[m_df['qr'] == 0, 'qtype_name'].value_counts().head(3)
m_top_src = m_df.loc[m_df['qr'] == 0, 'src_ip'].map(prefix_map).value_counts().head(3)
m_s1_share_peak = m_top_src.get(top_subnet_name, 0) / m_q * 100 if m_q else 0.0
print(f"[L] PROFIL MENIT PUNCAK {peak_minute} ({m_df.shape[0]:,} pesan = {m_df.shape[0] / median_lain:.1f}x median menit lain tanpa puncak):")
print(f"    Komposisi: {m_q:,} query + {m_r:,} respons | NXDOMAIN {m_nx:,} ({m_nx_rate:.2f}% respons)")
print("    qtype query teratas: " + ", ".join(f"{k} ({v:,})" for k, v in m_top_qtype.items()))
print("    Subnet sumber teratas: " + ", ".join(f"{anon(k)} ({v:,})" for k, v in m_top_src.items()))
if m_s1_share_peak < top_subnet_share:
    print(f"    Pangsa S1 pada menit puncak {m_s1_share_peak:.1f}% < pangsa keseluruhan "
          f"{top_subnet_share:.1f}% -> sumber terbesar keseluruhan belum tentu menjelaskan kenaikan menit puncak.")
else:
    print(f"    Pangsa S1 pada menit puncak {m_s1_share_peak:.1f}% >= pangsa keseluruhan "
          f"{top_subnet_share:.1f}% -> S1 mendominasi juga pada menit puncak.")
print("    Catatan: profil deskriptif; penyebab lonjakan perlu pemeriksaan lanjutan tim operasi.")

# ---------------------------------------------------------------
# [Langkah 19] Uraian lonjakan NXDOMAIN menit puncak + cakupan antrean
# Tujuan  : menghitung kontributor tambahan NXDOMAIN menit puncak
#           dibanding menit pembanding, dan cakupan investigasi
# Alasan  : pembanding = rata-rata menit penuh lainnya (tanpa menit
#           puncak) sehingga kenaikan terukur eksplisit; cakupan
#           menerjemahkan analisis menjadi beban kerja tim
# ---------------------------------------------------------------
nx_rows = df.loc[r_mask & (df['rcode'] == 3), ['ts_iso', 'dst_ip', 'qtype_name', 'qname']].copy()
nx_rows['minute'] = nx_rows['ts_iso'].dt.floor('min')
nx_rows = nx_rows[nx_rows['minute'].isin(steady_state_temporal.index)]
is_peak = nx_rows['minute'] == peak_minute
peak_nx = int(is_peak.sum())
delta_host, nx_summary = nx_delta(nx_rows, steady_state_temporal.index, peak_minute, group='dst_ip')
n_other = nx_summary['n_other_minutes']  # penyebut = seluruh menit pembanding, termasuk menit nol NX
base_nx_per_min = nx_summary['baseline_per_minute']
print('  - Ringkasan dekomposisi:', nx_summary)

nx_rows['reg'] = nx_rows['qname'].str.rstrip('.').str.lower().map(
    lambda q: psl.get_sld(q, strict=False) or q)
reg_peak = nx_rows.loc[is_peak, 'reg'].value_counts()
reg_base_mean = nx_rows.loc[~is_peak, 'reg'].value_counts() / max(n_other, 1)
union_reg = reg_peak.index.union(reg_base_mean.index)
delta_reg = (reg_peak.reindex(union_reg).fillna(0)
             - reg_base_mean.reindex(union_reg).fillna(0)).sort_values(ascending=False)

qt_peak = nx_rows.loc[is_peak, 'qtype_name'].value_counts()
qt_base_mean = nx_rows.loc[~is_peak, 'qtype_name'].value_counts() / max(n_other, 1)
union_qt = qt_peak.index.union(qt_base_mean.index)
delta_qt = (qt_peak.reindex(union_qt).fillna(0)
            - qt_base_mean.reindex(union_qt).fillna(0)).sort_values(ascending=False)

host_nx_all = nx_rows['dst_ip'].value_counts().sort_values(ascending=False)
cov5 = host_nx_all.head(5).sum() / max(len(nx_rows), 1) * 100
cov10 = host_nx_all.head(10).sum() / max(len(nx_rows), 1) * 100

print(f"[N] URAIAN LONJAKAN NX {peak_minute:%H:%M} vs rata-rata {n_other} menit lain "
      f"(NX {peak_nx:,} vs {base_nx_per_min:,.0f}/menit):")
host_label = {}

def hlabel(h):
    """Label H# untuk host penerima; unit analisis terpisah dari subnet S#."""
    if h not in host_label:
        host_label[h] = f'H{len(host_label) + 1}'
    return host_label[h]

print("  - Host penerima, top-5 delta: " + ", ".join(
    f"{hlabel(h)} +{d:,.0f}" for h, d in delta_host.head(5).items()))
_dh_total = peak_nx - base_nx_per_min
_top2h = delta_host.head(2).sum()
print(f"  - Dua host teratas (H1+H2): {_top2h:,.0f} dari {_dh_total:,.0f} tambahan bersih "
      f"({_top2h / max(_dh_total, 1e-9) * 100:.1f}%) - titik awal pemeriksaan, belum bukti penyebab.")
print("  - Registrable domain, top-5 delta: " + ", ".join(
    f"{k} +{d:,.0f}" for k, d in delta_reg.head(5).items()))
print("  - qtype, top-3 delta: " + ", ".join(
    f"{k} +{d:,.0f}" for k, d in delta_qt.head(3).items()))
print(f"  - Cakupan antrean: memeriksa 5 host teratas mencakup {cov5:.1f}% seluruh NXDOMAIN; "
      f"10 host mencakup {cov10:.1f}% (belum berarti sisanya bebas anomali).")
print(f"  - Rekonsiliasi: jumlah delta host = {delta_host.sum():,.0f} = selisih total "
      f"{peak_nx - base_nx_per_min:,.0f} | host delta negatif: {int((delta_host < 0).sum())}")

# Tabel keputusan: temuan -> tindakan -> indikator tindak lanjut
tc_rate_now = (tc_on_udp / total_udp_r) * 100 if total_udp_r else 0.0
keputusan = pd.DataFrame([
    {'Temuan': f"Beban terkonsentrasi: subnet {anon(top_subnet_name)} (S1) menyerap {top_subnet_share:.2f}% query, aktif {n_min_obs} menit",
     'Tindakan': 'Tim operasi memeriksa pola waktu, konfigurasi, dan kewajaran permintaan kelompok tersebut',
     'Indikator Tindak Lanjut': 'Pangsa query, laju query/menit, keberulangan lonjakan per interval'},
    {'Temuan': f"NXDOMAIN {nx_rate_global:.2f}% dari respons; sejumlah host penerima memiliki rasio NXDOMAIN tinggi (rasio top-5: {host_nx_rate_top5_lo:.0f}-{host_nx_rate_top5_hi:.0f}%); lima host teratas mencakup {cov5:.1f}% volume NXDOMAIN pada interval analisis",
     'Tindakan': 'Tim DNS memeriksa contoh QNAME, kemungkinan salah konfigurasi, dan pola permintaan host tersebut',
     'Indikator Tindak Lanjut': 'Jumlah dan rasio NXDOMAIN host pada periode pembanding'},
    {'Temuan': f"Truncation {tc_rate_now:.2f}% respons UDP (tc=1), terkonsentrasi pada {top2_desc}",
     'Tindakan': 'Tim DNS/jaringan memeriksa konfigurasi EDNS dan kesiapan TCP fallback',
     'Indikator Tindak Lanjut': 'Rasio truncation serta keberhasilan TCP dari pengukuran tambahan'},
])
print("\n[M] KEPUTUSAN PER TEMUAN (temuan -> tindakan -> indikator tindak lanjut):")
display(keputusan)

In [ ]:
# ---------------------------------------------------------------
# [Langkah 20] Ekspor agregat untuk dashboard HTML
# Tujuan  : menyerahkan hasil perhitungan Python sebagai satu-satunya sumber data visual
# Alasan  : dashboard (dashboard/index.html) hanya menata tampilan; tanpa angka ketik ulang,
#           ukuran kecil (agregat 29 menit + tabel), bukan 11,7 juta baris;
#           data.js format JS global agar file:// tetap berfungsi tanpa internet
# ---------------------------------------------------------------
import json as _json

_lbl_reg = {}
for _i, _d in enumerate(delta_reg.head(5).items()):
    _lbl_reg[_d[0]] = f'R{_i + 1}'  # label R# untuk visualisasi ringkas; nama asli pada tabel detail

_data = {
    'meta': {
        'judul': 'Analisis Operasional & Telemetri DNS Skala Besar',
        'zona_waktu': 'UTC',
        'periode_mulai': t_min.isoformat(),
        'periode_akhir': t_max.isoformat(),
        'n_menit_kanonik': int(n_min_obs),
        'n_pesan': int(len(df)),
        'n_query': int(total_q),
        'n_respons': int(total_r),
    },
    'kpi': [{'judul': t, 'nilai': v, 'cakupan': scope} for t, v, scope in kpi],
    'temporal': {
        'waktu': [m.strftime('%H:%M') for m in temporal.index],
        'waktu_iso': [m.isoformat() for m in temporal.index],
        'packets': temporal['packets'].tolist(),
        'nx': temporal['nx'].tolist(),
        'resp': temporal['responses'].tolist(),
        'nx_rate': [None if pd.isna(v) else float(v) for v in temporal['nx_rate']],
        'udp_resp': temporal['udp_resp'].tolist(),
        'udp_tc': temporal['udp_tc'].tolist(),
        'tc_ratio': [None if pd.isna(v) else float(v) for v in temporal['tc_ratio']],
        'median_lain': float(median_lain),
        'nx_rate_global': float(nx_rate_global),
        'tc_ratio_avg': float(tc_ratio_avg),
        'puncak': {'beban': peak_minute.strftime('%H:%M'),
                   'nx': nx_peak_minute.strftime('%H:%M'),
                   'tc': tc_peak_minute.strftime('%H:%M')},
    },
    'pareto': {
        'label': [anon(p) for p in prefix_df['Subnet Prefix']],
        'share': [float(v) for v in prefix_df['Share (%)']],
        'kumulatif': [float(v) for v in prefix_df['Kumulatif (%)']],
    },
    'prioritas_beban': [{'subnet': anon(_pfx), 'query': int(r['query']),
                          'share': float(r['share_pct']), 'menit': int(r['menit_aktif']),
                          'alasan': r['Alasan Prioritas']} for _pfx, r in f1.iterrows()],
    'prioritas_nx': [{'subnet': anon(_pfx), 'respons': int(r['respons']),
                      'nxdomain': int(r['nxdomain']), 'nx_rate': float(r['nx_rate_pct']),
                      'alasan': r['Alasan Prioritas']} for _pfx, r in f2.iterrows()],
    'bubble': {
        'titik': [{'label': anon(p), 'query': int(r['query']),
                   'nx_rate': None if pd.isna(r['nx_rate_pct']) else float(r['nx_rate_pct']),
                   'respons': int(r['respons']),
                   'nx_tinggi': bool(r['nx_rate_pct'] >= 25)}
                  for p, r in bub.iterrows()],
        'tak_terplot': [{'subnet': anon(p), 'query': int(r['query']), 'respons': int(r['respons']),
                         'nxdomain': int(r['nxdomain']),
                         'nx_rate': None if pd.isna(r['nx_rate_pct']) else float(r['nx_rate_pct'])}
                        for p, r in bub_lain.iterrows()] if len(bub_lain) else [],
    },
    'small_multiples': {
        'label': [anon(p) for p in top3p],
        'share': [float(prio.loc[p, 'share_pct']) for p in top3p],
        'waktu': [m.strftime('%H:%M') for m in pivot3.index],
        'seri': [pivot3[p].tolist() for p in top3p],
        'puncak': [{'menit': pivot3[p].idxmax().strftime('%H:%M'), 'nilai': float(pivot3[p].max())}
                   for p in top3p],
        'puncak_global': peak_minute.strftime('%H:%M'),
    },
    'dekomposisi': {
        'summary': {k: (float(v) if isinstance(v, (int, float, np.floating, np.integer)) else v)
                    for k, v in nx_summary.items()},
        'host': [{'label': hlabel(h), 'delta': float(d)} for h, d in delta_host.head(10).items()],
        'reg': [{'label': _lbl_reg[d], 'delta': float(v)} for d, v in delta_reg.head(5).items()],
        'reg_nilai': [float(d) for d in delta_reg.head(5).values],
        'qtype': [{'tipe': str(k), 'delta': float(d)} for k, d in delta_qt.head(3).items()],
        'cov5': float(cov5), 'cov10': float(cov10),
        'h1h2': {'nilai': float(_top2h), 'persen': float(_top2h / max(_dh_total, 1e-9) * 100),
                 'pembanding': float(_dh_total)},
        'nx_puncak': int(peak_nx), 'baseline_per_menit': float(base_nx_per_min),
    },
    'trunc_qtype': [{'tipe': str(k), 'rate': float(v), 'resp': int(n)}
                    for (k, v), n in zip(tq['rate'].items(), tq['resp'])],
    'heatmap': {
        'qtype': [str(x) for x in mat.index],
        'rcode': [str(rcode_map.get(int(c), str(c))) for c in mat.columns],
        'nilai': [[float(v) for v in baris] for baris in mat.values],
    },
    'komposisi': {
        'qtype': {'label': qt_labels, 'nilai': [int(v) for v in qt_vals]},
        'rcode': {'label': rc_labels, 'nilai': [int(v) for v in rc_vals]},
    },
}

def _sanitasi(o):
    """JSON-aman: NaN/NA menjadi null, numpy ke tipe bawaan, waktu ke ISO."""
    if isinstance(o, dict):
        return {str(k): _sanitasi(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [_sanitasi(v) for v in o]
    try:
        if pd.isna(o):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, (np.integer, int)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if float(o) != float(o) else float(o)
    if hasattr(o, 'isoformat'):
        return o.isoformat()
    return o

_bersih = _sanitasi(_data)
with open('../output/metrics_summary.json', 'w') as _f:
    _f.write(_json.dumps(_bersih, ensure_ascii=False, indent=1))
with open('../output/metrics_summary.js', 'w') as _f:
    _f.write('const PEDAS_DATA = ' + _json.dumps(_bersih, ensure_ascii=True) + ';\n')
print(f"Ekspor agregat dashboard OK: ../output/metrics_summary.json "
      f"({os.path.getsize('../output/metrics_summary.json'):,} byte) + data.js "
      f"({os.path.getsize('../output/metrics_summary.js'):,} byte)")



In [ ]:
from IPython.display import display
top2 = delta_host.head(2)

pemetaan_h1_h2 = pd.DataFrame({
    "Label": [hlabel(ip) for ip in top2.index],
    "IP dalam dataset": top2.index.tolist(),
    "Tambahan NXDOMAIN": top2.values,
})

display(pemetaan_h1_h2)